&nbsp;

# Other Sequencing related Metadata validated and Modified Process

- bulk RNA-seqを主に使用してきたが、CAGEなどの他のsequencing データも活用する方針なので、別途 こちらに記載するようにする
- 熱ストレス条件下のデータと、コントロール条件下のデータがプロジェクト内で取得でき、かつbulk RNA-seq以外のデータ
- ただし、ChIP-seqなどのデータは、別の notebook に記載している

&nbsp;

---

&nbsp;

- While bulk RNA-sequencing (RNA-seq) has been the primary methodology employed, other sequencing modalities such as Cap Analysis of Gene Expression (CAGE) will also be utilized; therefore, these shall be documented separately herein.
- This pertains to data acquired under both heat-stress and control conditions within the project, specifically excluding bulk RNA-seq data.
- However, data types such as Chromatin Immunoprecipitation Sequencing (ChIP-seq) are documented in a separate notebook.

In [1]:
import sys
import polars as pl
import datetime
import pathlib
from typing import Any
sys.path.append(str(pathlib.Path('../../').resolve()))
from scripts.metadata_pairing import create_paired_metadata
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_rows(4)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {execute_date}")

# create directory
pathlib.Path(f"../../Data/metadata/metadata_{VERSION}").mkdir(parents=True, exist_ok=True)

Execute Date: 260929


In [3]:
# drop columns list for iseq metadata
drop_columns_list = [
    "sra_aspera",
    "fastq_galaxy",
    "sra_galaxy",
    "fastq_ftp",
    "sra_ftp",
    "fastq_aspera"
]

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE165367 (SINGLE) 4sU-seq 

<!-- - ✅ DOWNLOAD -->
<!-- - ✅ TRIMMING -->
<!-- - ✅ QUANTIFICATION -->

&nbsp;

- validate date 2025-04-29
- [Heat shock induces premature transcript termination and reconfigures the human transcriptome](https://doi.org/10.1016/j.molcel.2022.01.007)
- 以前の研究でも使用したデータ (4sU-RNA標識法)
- コントロールと熱ストレス条件のデータが存在するため、すべてのデータを使用する
- [12_1_metadata_modified_true_control.ipynb](./12_1_metadata_modified_true_control.ipynb)から移動

&nbsp;

#### sample discription from PEP format csv

- `ctrl_4SU.mRNA-seq_rep3`
- `HS_4SU.mRNA-seq_rep1`

In [4]:
GSE165367_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE165367/GSE165367_PEP/GSE165367_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("KAPA mRNA HyperPrep kit").alias("rna_library"), # add rna_library column (Roche)
    pl.lit("https://doi.org/10.1016/j.molcel.2022.01.007").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("135 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE165367 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE165367_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE165367_filtered = GSE165367[[s.name for s in GSE165367 if not (s.null_count() == GSE165367.height)]]

# display(GSE165367_PEP)

In [5]:
GSE165367_combined = GSE165367_PEP.join(
    GSE165367_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE165367_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE165367_modified_metadata.tsv",
    separator='\t'
)

# display(GSE165367_combined)

In [6]:
# Manual defined pairs
defined_pairs_gse165367_4sU_seq = [
    ("SRR13510346", "SRR13510343"), # heat-stress sample vs control sample
    ("SRR13510347", "SRR13510344"), # heat-stress sample vs control sample
    ("SRR13510348", "SRR13510345") # heat-stress sample vs control sample
]

paired_metadata_df_gse165367_4sU_seq = create_paired_metadata(
    combined_df=GSE165367_combined, 
    defined_sample_pairs=defined_pairs_gse165367_4sU_seq,
    sample_id_column_name="accession" # default value, but specified to clarify
)


if paired_metadata_df_gse165367_4sU_seq is not None:
    paired_metadata_df_gse165367_4sU_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE165367_4sU_seq_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse165367_4sU_seq)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

# display(paired_metadata_df_gse165367_4sU_seq)

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,…,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR13510346""","""SRR13510343""","""hs_4su_mrna-seq_rep1 vs ctrl_4su_mrna-seq_rep1""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR13510346 vs SRR13510343""","""SRX9921749 vs SRX9921746""","""HS_4SU.mRNA-seq_rep1 vs ctrl_4SU.mRNA-seq_rep1""","""GSM5032231 vs GSM5032228""","""Public on Jan 24 2022""","""Jan 22 2021""","""Jan 24 2022""","""SRA""","""1""","""MRC5-VA cell line""","""Homo sapiens""","""9606""","""polyA RNA""","""mRNA-Seq, poly(A)+ mRNA""","""GPL20301""","""Richard,James,Mitter""","""Bioinformatics & Biostatistics""","""The Francis Crick Institute""","""1 Midland Road""","""London""","""LONDON""","""NW1 1AT""","""United Kingdom""","""Illumina HiSeq 4000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5032231 vs GSM5032228""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5032nnn/GSM5032231/suppl/GSM5032231_HS_4SU.mRNA-seq_rep1.bigWig vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5032nnn/GSM5032228/suppl/GSM5032228_ctrl_4SU.mRNA…","""GSE165367, GSE165368""",…,"""135 min.""","""unknown (maybe 37 C)""","""SRR13510346 vs SRR13510343""","""Illumina HiSeq 4000 sequencing: GSM5032231: HS_4SU.mRNA-seq_rep1 Homo sapiens RNA-Seq vs Illumina HiSeq 4000 sequencing: GSM5032228: ctrl_4SU.mRNA-seq_rep1 Homo sapiens RNA-Seq""","""Heat shock-mediated global gene downregulation is achieved by premature transcription termination [4SU_mRNA-seq]""","""SRA1186922""","""cDNA""","""2022-02-24""","""2049836161 vs 2262294235""","""2923067836 vs 3246878490""","""ILLUMINA""","""Generic""","""HS_4SU.mRNA-seq_rep1 vs ctrl_4SU.mRNA-seq_rep1""","""9fda9bd41d6be5d209a19d031931a875 vs 52d797fabed56fb3c2d40f34c777d630""","""SRP302922""","""57869116 vs 63650092""","""Heat shock-mediated global gene downregulation is achieved by premature transcription termination [4SU_mRNA-seq]""","""HS_4SU.mRNA-seq_rep1 vs ctrl_4SU.mRNA-seq_rep1""","""Illumina HiSeq 4000""","""Illumina HiSeq 4000 sequencing: GSM5032231: HS_4SU.mRNA-seq_rep1 Homo sapiens RNA-Seq vs Illumina HiSeq 4000 sequencing: GSM5032228: ctrl_4SU.mRNA-seq_rep1 Homo sapiens RNA-Seq""","""39e7665d8e8e00e1d1812488d5cad2ad vs 09fd01762e9a820e18f64407fd85655b""","""5844780716 vs 6428659292""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM5032231 vs GSM5032228""","""GSM5032231_r1 vs GSM5032228_r1""","""RNA was extracted using the RNeasy kit (Qiagen, 74104) according to the manufacturer's instruction including the on-column DNase

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE165364 (SINGLE) TTchem-seq 

<!-- - ✅ DOWNLOAD -->
<!-- - ✅ TRIMMING -->
<!-- - ✅ QUANTIFICATION -->

&nbsp;

- validate date 2025-04-29
- [Heat shock induces premature transcript termination and reconfigures the human transcriptome](https://doi.org/10.1016/j.molcel.2022.01.007)
- 以前の研究でも使用したデータ (TTchem-seq)
- コントロールと熱ストレス条件のデータが存在するため、すべてのデータを使用する予定...
- しかしながら、nacsent RNA のバイアスを考慮できているかが気になるので後でまた使用するかどうか検討する
- [Using TTchem-seq for profiling nascent transcription and measuring transcript elongation](https://doi.org/10.1038/s41596-019-0262-3) などの論文を見ると、STARのマッピングなどを使用しているため、おそらく発現定量の方法はRNA-seqと同じ？
- 120分の実験を行ったあと、追加で30,60,120min.の実験をしている
- 異なるペアで熱ストレスの条件が異なるので後で手入力
- [./12_1_metadata_modified_true_control.ipynb](./12_1_metadata_modified_true_control.ipynb)から処理を移動

&nbsp;

#### sample discription from PEP format csv

- `mrc5va_ctrl_ttchem-seq_rep1`
- `mrc5va_hs120min_ttchem-seq_rep1`
- ...

In [7]:
GSE165364_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE165364/GSE165364_PEP/GSE165364_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("KAPA mRNA HyperPrep kit").alias("rna_library"), # add rna_library column (Roche)
    pl.lit("https://doi.org/10.1016/j.molcel.2022.01.007").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    # pl.lit("unknown").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE165364 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE165364_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE165364_filtered = GSE165364[[s.name for s in GSE165364 if not (s.null_count() == GSE165364.height)]]

# display(GSE165364_PEP)

In [8]:
GSE165364_combined = GSE165364_PEP.join(
    GSE165364_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE165364_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE165364_modified_metadata.tsv",
    separator='\t'
)

# display(GSE165364_combined)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE165366 (SINGLE) FP-TTchem-seq 

<!-- - ✅ DOWNLOAD -->
<!-- - ✅ TRIMMING -->
<!-- - ✅ QUANTIFICATION -->

<!-- &nbsp; -->

- validate date 2025-04-29
- [Heat shock induces premature transcript termination and reconfigures the human transcriptome](https://doi.org/10.1016/j.molcel.2022.01.007)
- 以前の研究でも使用したデータ (FP:Flavopiridol.TTchem-seq)
- コントロールと熱ストレス条件のデータが存在するため、すべてのデータを使用する予定...
- しかしながら、nacsent RNA のバイアスを考慮できているかがきになるので後でまた使用するかどうか検討する
- [./12_1_metadata_modified_true_control.ipynb](./12_1_metadata_modified_true_control.ipynb)から処理を移動

#### sample discription from PEP format csv

- `2h HS, followed by 15min of treatment with 300nM Flavopiridol (including 10 min 1mM 4-thiouridine pulse)`
- `2h HS, followed by 20min of treatment with 300nM Flavopiridol (including 10 min 1mM 4-thiouridine pulse)`
- ...

In [9]:
GSE165366_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE165366/GSE165366_PEP/GSE165366_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("KAPA mRNA HyperPrep kit").alias("rna_library"), # add rna_library column (Roche)
    pl.lit("https://doi.org/10.1016/j.molcel.2022.01.007").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("120 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE165366 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE165366_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE165366_filtered = GSE165366[[s.name for s in GSE165366 if not (s.null_count() == GSE165366.height)]]

# display(GSE165366_PEP)

In [10]:
GSE165366_combined = GSE165366_PEP.join(
    GSE165366_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE165366_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE165366_modified_metadata.tsv",
    separator='\t'
)

# display(GSE165366_combined)

In [11]:
# Manual defined pairs
defined_pairs_gse165366_fp_ttchem_seq = [
    ("SRR13510331", "SRR13510319"), # heat-stress sample vs control sample
    ("SRR13510332", "SRR13510320"), # heat-stress sample vs control sample
    ("SRR13510333", "SRR13510321"), # heat-stress sample vs control sample
    ("SRR13510334", "SRR13510322"), # heat-stress sample vs control sample
    ("SRR13510335", "SRR13510323"), # heat-stress sample vs control sample
    ("SRR13510336", "SRR13510324"), # heat-stress sample vs control sample
    ("SRR13510337", "SRR13510325"), # heat-stress sample vs control sample
    ("SRR13510338", "SRR13510326"), # heat-stress sample vs control sample
    ("SRR13510339", "SRR13510327"), # heat-stress sample vs control sample
    ("SRR13510340", "SRR13510328"), # heat-stress sample vs control sample
    ("SRR13510341", "SRR13510329"), # heat-stress sample vs control sample
    ("SRR13510342", "SRR13510330")  # heat-stress sample vs control sample
]

paired_metadata_df_gse165366_fp_ttchem_seq = create_paired_metadata(
    combined_df=GSE165366_combined, 
    defined_sample_pairs=defined_pairs_gse165366_fp_ttchem_seq,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse165366_fp_ttchem_seq is not None:
    paired_metadata_df_gse165366_fp_ttchem_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE165366_fp_ttchem_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse165366_fp_ttchem_seq)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,…,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR13510331""","""SRR13510319""","""hs_fp_ttchem-seq_rep1 vs ctrl_fp_ttchem-seq_rep1""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR13510331 vs SRR13510319""","""SRX9921734 vs SRX9921722""","""HS_FP.TTchem-seq_rep1 vs ctrl_FP.TTchem-seq_rep1""","""GSM5032216 vs GSM5032204""","""Public on Jan 24 2022""","""Jan 22 2021""","""Jan 24 2022""","""SRA""","""1""","""MRC5-VA cell line""","""Homo sapiens""","""9606""","""total RNA""","""FP/TTchem-seq, Nascent RNA""","""GPL20301""","""Richard,James,Mitter""","""Bioinformatics & Biostatistics""","""The Francis Crick Institute""","""1 Midland Road""","""London""","""LONDON""","""NW1 1AT""","""United Kingdom""","""Illumina HiSeq 4000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5032216 vs GSM5032204""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5032nnn/GSM5032216/suppl/GSM5032216_HS_FP.TTchem-seq_rep1.bigWig vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5032nnn/GSM5032204/suppl/GSM5032204_ctrl_FP.TTch…","""GSE165366, GSE165368""",…,"""120 min.""","""unknown (maybe 37 C)""","""SRR13510331 vs SRR13510319""","""Illumina HiSeq 4000 sequencing: GSM5032216: HS_FP.TTchem-seq_rep1 Homo sapiens RNA-Seq vs Illumina HiSeq 4000 sequencing: GSM5032204: ctrl_FP.TTchem-seq_rep1 Homo sapiens RNA-Seq""","""Heat shock-mediated global gene downregulation is achieved by premature transcription termination [FP_TTchem-seq]""","""SRA1186912""","""cDNA""","""2022-02-24""","""2399353716 vs 2392279455""","""3337864604 vs 3239674491""","""ILLUMINA""","""Generic""","""HS_FP.TTchem-seq_rep1 vs ctrl_FP.TTchem-seq_rep1""","""e3b2a00e8542dbef66ef1c1e853526fe vs 5b5f14ee19160fd05c6a390b532feaf4""","""SRP302921""","""59232897 vs 58747598""","""Heat shock-mediated global gene downregulation is achieved by premature transcription termination [FP_TTchem-seq]""","""HS_FP.TTchem-seq_rep1 vs ctrl_FP.TTchem-seq_rep1""","""Illumina HiSeq 4000""","""Illumina HiSeq 4000 sequencing: GSM5032216: HS_FP.TTchem-seq_rep1 Homo sapiens RNA-Seq vs Illumina HiSeq 4000 sequencing: GSM5032204: ctrl_FP.TTchem-seq_rep1 Homo sapiens RNA-Seq""","""01474b9590916a56f3a51f919609703f vs b158343392614f05362ee10f5ce2f3f5""","""5982522597 vs 5933507398""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM5032216 vs GSM5032204""","""GSM5032216_r1 vs GSM5032204_r1""","""RNA was extracted by TRIzol (Thermo Fisher Scientific, 15596026) according to the manufacturer's instructions. 

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE140053 (SINGLE) SLAM-seq

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-04-29
- [Stress-induced nuclear condensation of NELF drives transcriptional downregulation](https://doi.org/10.1016/j.molcel.2021.01.016)
- これも以前の研究で使用したデータ (SLAM-seq)
- 定量できるかわからないが、これも集めよう...
- ChiP-seqのデータは取り除く ([./12_8_metadata_chipseq.ipynb](./12_8_metadata_chipseq.ipynb) で処理)
- `hs_nelfa_didr_rep3` のデータがないので、ペアとなる`nhs_nelfa_wt_rep3` は今回は使用しない
- [./12_1_metadata_modified_true_control.ipynb](./12_1_metadata_modified_true_control.ipynb)から移動

In [12]:
GSE140053_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE140053/GSE140053_PEP/GSE140053_PEP_raw.csv",
    separator=','
).filter(
    pl.col("protocol").str.contains(r'other') # remove ChIP-seq data
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("Quantseq 3’ mRNA-Seq Library Prep Kit").alias("rna_library"), # add rna_library column (Lexogen)
    pl.lit("https://doi.org/10.1016/j.molcel.2021.01.016").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("30 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
)

# metadata from iseq
GSE140053 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE140053_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE140053_filtered = GSE140053[[s.name for s in GSE140053 if not (s.null_count() == GSE140053.height)]]

# display(GSE140053_PEP)

In [13]:
GSE140053_combined = GSE140053_PEP.join(
    GSE140053_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE140053_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE140053_modified_slam_seq_metadata.tsv",
    separator='\t'
)

# display(GSE140053_combined)

In [14]:
# Manual defined pairs
defined_pairs_gse140053 = [
    ("SRR11067753", "SRR11067765"),
    ("SRR11067754", "SRR11067766"),
    ("SRR11067755", "SRR11067767"),
    ("SRR11067756", "SRR11067768"),
    ("SRR11067757", "SRR11067769"),
    ("SRR11067758", "SRR11067770"),
    ("SRR12801710", "SRR12801715"),
    ("SRR12801711", "SRR12801716"),
    ("SRR12801712", "SRR12801718"),
    ("SRR12801713", "SRR12801719"),
    ("SRR12801714", "SRR12801720") 
]

paired_metadata_df_gse140053_slam_seq = create_paired_metadata(
    combined_df=GSE140053_combined, 
    defined_sample_pairs=defined_pairs_gse140053,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse140053_slam_seq is not None:
    paired_metadata_df_gse140053_slam_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE140053_slam_seq_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse140053_slam_seq)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

# display(paired_metadata_df_gse140053_slam_seq)

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,…,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR11067753""","""SRR11067765""","""hela_hs_siscr_slam-seq_rep1 vs hela_nhs_siscr_slam-seq_rep1""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR11067753 vs SRR11067765""","""SRX7708034 vs SRX7708046""","""HeLa_HS_siSCR_Slam-seq_REP1 vs HeLa_NHS_siSCR_Slam-seq_REP1""","""GSM4306529 vs GSM4306541""","""Public on Feb 05 2021""","""Feb 11 2020""","""Feb 06 2021""","""SRA""","""1""","""HeLa""","""Homo sapiens""","""9606""","""Heat-shock and labeling was carried out 24 hours after the change of media (48 hours after transfection and/or knockdown) in the following manner: Cells were subject to heat shock at 43 degrees Celsiu…","""HeLa cells were plated, and after 24 hours for experiment (1) NELF-A Wildtype was knocked down using Lipofectamine RNAiMAX with concurrent transfection with either NELF-A WT or NELF-A ΔIDR using Lipof…","""total RNA""","""RNA was extracted using Trizol per manufactures protocol, with addition of DTT, and protection from light during the procedure to prevent S-S cross-binding., 5 ug of RNA was used for SLAMseq modificat…","""null""","""3'UTR annotations were taken from Gencode annotation release 31 and merged on a gene level. Adapters and polyA stretches were trimmed from raw reads using fastp. Trimmed reads were further processed w…","""GPL18573""","""Barbara,,Hummel""","""Stübeweg 51""","""Freiburg""","""79108""","""Germany""","""Illumina NextSeq 500""","""other""","""transcriptomic""","""OTHER""","""GSM4306529 vs GSM4306541""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4306nnn/GSM4306529/suppl/GSM4306529_HeLa_HS_siSCR_REP1_R1.fastq_slamdunk_mapped_filtered_tcount.tsv.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4306nnn/GS…",…,"""30 min.""","""unknown (maybe 37 C)""","""SRR11067753 vs SRR11067765""","""NextSeq 500 sequencing: GSM4306529: HeLa_HS_siSCR_Slam-seq_REP1 Homo sapiens OTHER vs NextSeq 500 sequencing: GSM4306541: HeLa_NHS_siSCR_Slam-seq_REP1 Homo sapiens OTHER""","""Stress-induced nuclear condensation of NELF drives global transcriptional downregulation""","""SRA992219""","""other""","""2021-02-07""","""1084070193 vs 1070100006""","""1068781563 vs 1042343011""","""ILLUMINA""","""Generic""","""HeLa_HS_siSCR_Slam-seq_REP1 vs HeLa_NHS_siSCR_Slam-seq_REP1""","""9b5315e729dab36b379fec554e4f68a4 vs 3eabcd3284910d1b49b7a12e00f3a0a9""","""SRP228821""","""27725226 vs 27550481""","""Stress-induced nuclear condensation of NE

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE233555 Ribo-Calibration (PAIRED)

- validation date: 2025-05-02
- [Calibrated ribosome profiling assesses the dynamics of ribosomal flux on transcripts](https://doi.org/10.1038/s41467-024-51258-0)
- 
- `For heat shock treatment, HEK293 cells cultured in 6-well plates were transferred to a water bath set at 43 °C and incubated for 0.5, 1, or 2 h. For recovery experiments, cells were returned to 37 °C and cultured for 0.5, 1, or 4 h.`
- リボソームのプロファイリング手法に関連する研究
- Homo sapiens のRNA-seqのデータを取得
- おそらく `RNA_HS...` で始まるデータが関連しているのでフィルタリング
- [./12_1_metadata_modified_true_control.ipynb](./12_1_metadata_modified_true_control.ipynb)から処理を移動

In [15]:
GSE233555_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE233555/GSE233555_PEP/GSE233555_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_organism_ch1") == "Homo sapiens") &
    (pl.col("sample_library_strategy") == "RNA-Seq") &
    (pl.col("sample_title").str.contains(r'RNA_HS_')) # filter Heat-stress related RNA-sequencing Data
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq Stranded mRNA Library Prep Kit").alias("rna_library"), # add rna_library column (Illumina)
    pl.lit("https://doi.org/10.1038/s41467-024-51258-0").alias("reference_doi"), # add DOI of the reference paper column (Nature Communications)
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    # pl.lit("30 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE233555 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE233555_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE233555_filtered = GSE233555[[s.name for s in GSE233555 if not (s.null_count() == GSE233555.height)]]

# display(GSE233555_PEP)

In [16]:
GSE233555_combined = GSE233555_PEP.join(
    GSE233555_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE233555_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE233555_ribo_calibration_modified_metadata.tsv",
    separator='\t'
)

# display(GSE233555_combined)

In [17]:
# Manual defined pairs
defined_pairs_gse233555 = [
    ("SRR24746782", "SRR24746784"), # RNA_HS_0.5h vs RNA_HS_CT1
    ("SRR24746781", "SRR24746784"), # RNA_HS_1h vs RNA_HS_CT1
    ("SRR24746780", "SRR24746784"), # RNA_HS_2h vs RNA_HS_CT1
    ("SRR24746763", "SRR24746783"), # RNA_HS_2_0.5h vs RNA_HS_CT2
    ("SRR24746762", "SRR24746783"), # RNA_HS_2_1h vs RNA_HS_CT2
    ("SRR24746761", "SRR24746783")  # RNA_HS_2_4h vs RNA_HS_CT2
]

paired_metadata_df_gse233555_ribo_calibration = create_paired_metadata(
    combined_df=GSE233555_combined, 
    defined_sample_pairs=defined_pairs_gse233555,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse233555_ribo_calibration is not None:

    paired_metadata_df_gse233555_ribo_calibration.write_csv(
        "../../Data/metadata/metadata_paired/GSE233555_ribo_calibration_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse233555_ribo_calibration)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,…,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,cell_type_right,base_count,library_strategy_right,library_source,library_layout,datahub,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,strain,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR24746782""","""SRR24746784""","""rna_hs_0_5 vs rna_hs_ct1""","""null""","""null""","""null""","""null""","""null""","""null""","""RNA_HS_0.5 vs RNA_HS_CT1""","""GSM7430271 vs GSM7430269""","""Public on Jul 10 2024""","""May 26 2023""","""Jul 10 2024""","""SRA""","""1""","""cultures""","""Homo sapiens""","""9606""","""total RNA""","""HEK cells were cultured in a 10 cm dish in preparation for the RocA treatment, and in a 6-well plate for the heat shock treatment. The cells were then lysed in 500 µl and 100 µl of lysis buffer., Hum…","""RNAseq_and_rRNA_depletion_with_Ribo-Zero_Ribo-Calibration""","""GPL20795""","""Shintaro,,Iwasaki""","""shintaro.iwasaki@riken.jp""","""RNA Systems Biochemistry Laboratory""","""CPR""","""RIKEN""","""2-1, Hirosawa""","""Wako city""","""351-0198""","""Japan""","""HiSeq X Ten""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM7430271 vs GSM7430269""",…,"""2024-07-11""","""1028372965 vs 1094920901""","""483792922;580125521 vs 623626517;715483215""","""ILLUMINA""","""missing""","""Generic""","""RNA_HS_0.5 vs RNA_HS_CT1""","""6a18996959f400dd71a62f5b62f4b8ce;03af8dc9907413e7dcab9a4e9c9bf6b7 vs f4c9bde191d69ba05e15e246d574dbdb;9d6cf382f7c832663fe9ea7e668e924a""","""SRP439789""","""9227732 vs 10036358""","""Calibrated ribosome profiling assesses the dynamics of ribosomal flux on transcripts""","""RNA_HS_0.5 vs RNA_HS_CT1""","""HiSeq X Ten""","""HiSeq X Ten sequencing: GSM7430271: RNA_HS_0.5 Homo sapiens RNA-Seq vs HiSeq X Ten sequencing: GSM7430269: RNA_HS_CT1 Homo sapiens RNA-Seq""","""7d6ca40734c33e8cb5fdd10fda98ec66 vs cecf4b1f5ce8ba8a6a2036c29a283f29""","""Human Embryonic Kidney cells293""","""2554679284 vs 2765058935""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""null""","""public""","""GSM7430271 vs GSM7430269""","""GSM7430271_r1 vs GSM7430269_r1""","""HEK cells were cultured in a 10 cm dish in preparation for the RocA treatment, and in a 6-well plate for the heat shock treatment. The cells were then lysed in 500 µl and 100 µl of lysis buffer. Human…","""missing""","""GSM7430271_r1 vs GSM7430269_r1""","""HEK293""","""GSM7430271 vs GSM7430269""","""null""","""2024-07-11""","""null""","""PRJNA976655""","""SRS17833492 vs SRS17833490""","""2024-07-11""","""GSE233555""","""cultures"""
"""SRR24746781""","""SRR24746784""","""rna_hs_1 vs rna_hs_ct1""","""null""","""null""","""null""","""null""","""null""","""null""","""RNA_HS_1 vs RNA_HS_CT1""","""GSM7430272 vs GSM7430269""","""Public on Jul 10

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE124609 (SINGLE) Ribo-seq

<!-- - ✅ DOWNLOAD -->
<!-- - ✅ TRIMMING -->
<!-- - ✅ QUANTIFICATION -->

&nbsp;

- validate date 2025-04-29
- [Cellular proteostasis decline in human senescence](https://doi.org/10.1073/pnas.2018138117)
- コントロールと熱ストレス条件のデータが存在するため、すべてのデータを使用する予定...
- Ribo-seqのデータの処理を[./12_1_metadata_modified_true_control.ipynb](./12_1_metadata_modified_true_control.ipynb)から移動

#### sample discription from PEP format csv

- `Young1_RNA_Seq`
- `Young_HS1_RNA_Seq`
- ...

In [18]:
GSE124609_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE124609/GSE124609_PEP/GSE124609_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("Illumina kit (TruSeq RNA)").alias("rna_library"), # add rna_library column (Illumina: from Article)
    pl.lit("https://doi.org/10.1073/pnas.2018138117").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("44 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("120 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# ribo-seq
GSE124609_PEP_ribo_seq = GSE124609_PEP.filter(
    pl.col("sample_name").str.contains(r'ribo_seq')
)


# metadata from iseq
GSE124609 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE124609_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE124609_filtered = GSE124609[[s.name for s in GSE124609 if not (s.null_count() == GSE124609.height)]]

# display(GSE124609_PEP_rna_seq)
# display(GSE124609_PEP_ribo_seq)

In [19]:
GSE124609_ribo_seq_combined = GSE124609_PEP_ribo_seq.join(
    GSE124609_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE124609_ribo_seq_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE124609_ribo_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE124609_ribo_seq_combined)

In [20]:
# Manual defined pairs
defined_pairs_gse124609_ribo_seq = [
    ("SRR8392707", "SRR8392705"),
    ("SRR8392708", "SRR8392706"),
    ("SRR8392711", "SRR8392709"),
    ("SRR8392712", "SRR8392710")
]

paired_metadata_df_gse124609_ribo_seq = create_paired_metadata(
    combined_df=GSE124609_ribo_seq_combined, 
    defined_sample_pairs=defined_pairs_gse124609_ribo_seq,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse124609_ribo_seq is not None:
    paired_metadata_df_gse124609_ribo_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE124609_paired_ribo_seq_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse124609_ribo_seq)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,…,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR8392707""","""SRR8392705""","""young_hs1_ribo_seq vs young1_ribo_seq""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR8392707 vs SRR8392705""","""SRX5202418 vs SRX5202416""","""Young_HS1_Ribo_Seq vs Young1_Ribo_Seq""","""GSM3537592 vs GSM3537590""","""Public on Dec 09 2020""","""Jan 03 2019""","""Dec 09 2020""","""SRA""","""1""","""WI38 cells""","""Homo sapiens""","""9606""","""total RNA""","""GPL11154""","""Reut,,Shalgi""","""reutshalgi@technion.ac.il""","""Faculty of Medicine""","""Technion""","""1 Efron st.""","""Haifa""","""31096""","""Israel""","""Illumina HiSeq 2000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM3537592 vs GSM3537590""","""NONE""","""GSE124609""","""0""",…,"""unknown (maybe 37 C)""","""SRR8392707 vs SRR8392705""","""Illumina HiSeq 2000 sequencing: GSM3537592: Young_HS1_Ribo_Seq Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM3537590: Young1_Ribo_Seq Homo sapiens RNA-Seq""","""Cellular proteostasis collapse in mammalian senescence""","""SRA830119""","""cDNA""","""2021-01-04""","""1579560389 vs 2889479403""","""1714109373 vs 3050513189""","""ILLUMINA""","""Generic""","""Young_HS1_Ribo_Seq vs Young1_Ribo_Seq""","""299284e444580d857166102e41218d64 vs 8e760edd4dc586718214fd40f13b41f5""","""SRP175158""","""60450688 vs 91019238""","""Cellular proteostasis collapse in mammalian senescence""","""Young_HS1_Ribo_Seq vs Young1_Ribo_Seq""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM3537592: Young_HS1_Ribo_Seq Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM3537590: Young1_Ribo_Seq Homo sapiens RNA-Seq""","""7e42c0d7fc9be29b293da93868adc7fd vs 20f09ee5e79dcc85edd3fadeff6cf4a3""","""human lung fibroblasts""","""3753949314 vs 6631519393""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM3537592 vs GSM3537590""","""GSM3537592_r1 vs GSM3537590_r1""","""RNA-Seq libraries: RNA was isolated from collected samples using the TRIZOL reagent (Thermo) following treatment with Turbo DNase and phenol chloroform precipitation. RNA-seq: After quality control, t…","""GSM3537592 vs GSM3537590""","""GEO""","""WI38""","""2021-01-04""","""PRJNA512837""","""SRS4207569 vs SRS4207567""","""2021-01-04""","""GSE124609"""
"""SRR8392708""","""SRR8392706""","""young_hs2_ribo_seq vs young2_ribo_seq""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR8392708 vs SRR8392706""","""SRX5202419 vs SRX5202417""","""Young_HS2_Ribo_Seq

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE32060 (SINGLE) Ribo-seq

<!-- - ✅ DOWNLOAD -->
<!-- - ✅ TRIMMING -->
<!-- - ✅ QUANTIFICATION -->

&nbsp;

- validation date: 2025-05-02
- [Widespread Regulation of Translation by Elongation Pausing in Heat Shock](https://doi.org/10.1016/j.molcel.2012.11.028)
- 以前の研究でも使用した研究
- ペアごとに熱ストレスの条件に違うので、後で手入力
- [./12_1_metadata_modified_true_control.ipynb](./12_1_metadata_modified_true_control.ipynb)から処理を移動

In [21]:
GSE32060_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE32060/GSE32060_PEP/GSE32060_PEP_raw.csv",
    separator=','
).filter(
    pl.col("organism") == "Homo sapiens"
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column (Illumina)
    pl.lit("https://doi.org/10.1016/j.molcel.2012.11.028").alias("reference_doi"), # add DOI of the reference paper column
    # pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    # pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE32060 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE32060_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE32060_filtered = GSE32060[[s.name for s in GSE32060 if not (s.null_count() == GSE32060.height)]]

# display(GSE32060_PEP)

In [22]:
GSE32060_combined = GSE32060_PEP.join(
    GSE32060_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE32060_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE32060_ribo_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE32060_combined)

In [23]:
# Manual defined pairs
defined_pairs_gse32060 = [
    ("SRR648668", "SRR648667"),
    ("SRR648670", "SRR648669")
]

paired_metadata_df_gse32060_ribo_seq = create_paired_metadata(
    combined_df=GSE32060_combined, 
    defined_sample_pairs=defined_pairs_gse32060,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse32060_ribo_seq is not None:
    paired_metadata_df_gse32060_ribo_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE32060_ribo_seq_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse32060_ribo_seq)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,…,rna_library,reference_doi,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR648668""","""SRR648667""","""293t_gfp_2hrs_severe_heat_shock vs 293t_gfp""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR648668 vs SRR648667""","""SRX217029 vs SRX217028""","""293T_GFP_2hrs_severe_Heat_Shock vs 293T_GFP""","""GSM1063576 vs GSM1063575""","""Public on Jan 15 2013""","""Jan 14 2013""","""May 15 2019""","""SRA""","""1""","""293T cells""","""Homo sapiens""","""9606""","""DMEM+10%FBS""","""total RNA""","""Riboseq - described in Shalgi et al. 2013. similar to Ingolia et al. Science, 2009""","""GPL9115""","""Reut,,Shalgi""","""reutshalgi@technion.ac.il""","""Faculty of Medicine""","""Technion""","""1 Efron st.""","""Haifa""","""31096""","""Israel""","""Illumina Genome Analyzer II""","""other""","""transcriptomic""","""RNA-Seq""","""GSM1063576 vs GSM1063575""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1063nnn/GSM1063576/suppl/GSM1063576_GFP_HS.fastq.bed.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1063nnn/GSM1063575/suppl/GSM1063575_GFP.fastq.bed.gz""",…,"""unknown""","""https://doi.org/10.1016/j.molcel.2012.11.028""","""unknown (maybe 37 C)""","""SRR648668 vs SRR648667""","""Illumina Genome Analyzer II sequencing: GSM1063576: 293T_GFP_2hrs_severe_Heat_Shock Homo sapiens RNA-Seq vs Illumina Genome Analyzer II sequencing: GSM1063575: 293T_GFP Homo sapiens RNA-Seq""","""Widespread regulation of translation by elongation pausing in heat shock""","""SRA064749""","""other""","""2020-06-05 vs 2019-12-27""","""2133784656 vs 1149945443""","""2722139656 vs 1481422431""","""ILLUMINA""","""Generic""","""293T_GFP_2hrs_severe_Heat_Shock vs 293T_GFP""","""f92bb28f26fd9e991f74ed6e94c7de82 vs 9bac2cbf4e9680f0797c96206d72c9cc""","""SRP017942""","""95471801 vs 50184481""","""Widespread regulation of translation by elongation pausing in heat shock""","""293T_GFP_2hrs_severe_Heat_Shock vs 293T_GFP""","""Illumina Genome Analyzer II""","""Illumina Genome Analyzer II sequencing: GSM1063576: 293T_GFP_2hrs_severe_Heat_Shock Homo sapiens RNA-Seq vs Illumina Genome Analyzer II sequencing: GSM1063575: 293T_GFP Homo sapiens RNA-Seq""","""fbfc6e9e57f7a5c5f99623eab3c9d8c7 vs dbc5f68d351c25f1123bf2656a77708c""","""3436984836 vs 1806641316""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM1063576 vs GSM1063575""","""GSM1063576_r1 vs GSM1063575_r1""","""GSM1063576_1 vs GSM1063575_1""","""GEO""","""293T cells""","""2013-11-11""","""PRJNA147613""","""SRS385796 vs SRS385795""","

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE234710 (PAIRED) (BRB-seq)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-03
- validate date: 2026-05-04
- Bulk RNA barcoding sequencing (BRB) という手法だが、メタデータを見ると、`RNA-Seq` となっている
- [Transcriptional landscape of human keratinocyte models exposed to 60-GHz millimeter-waves](https://doi.org/10.1016/j.tiv.2024.105808)
- `Sham unexposed cells(Sham)`, `millimeter waves (MMW) exposed Cells (Expo)`, `heat shock control (HSC)`の3つの実験条件 + `HEK, NHEK, HaCaT` の3つの細胞種の条件
- 温度条件は論文の [Table 1](https://www.sciencedirect.com/science/article/pii/S0887233324000389?via%3Dihub#t0005), Table 3 に記載されている
- `sham` は 36℃前後であることから、これをコントロールとして使用できると考えられる
- ~~熱を加えている、という側面があるため、全てのデータを使用する~~
- **`heat shock control (HSC)` と `sham` のみ 収集する方針に転換**
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から移動

In [24]:
GSE234710_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE234710/GSE234710_PEP/GSE234710_PEP_raw.csv",
    separator=','
).filter(
    (~pl.col("sample_title").str.contains("Expo"))
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column from GEO webpage
    pl.lit("https://doi.org/10.1016/j.tiv.2024.105808").alias("reference_doi"), # add DOI of the reference paper column

    # stress treatment temperature
    pl.when(
        (pl.col("sample_name").str.contains("hacat_3h_hsc_5"))
    ).then(
        pl.lit("36.55 ± 0.53 C") # Table 3
    ).when(
        (pl.col("sample_name").str.contains("hacat_3h_hsc_10_"))
    ).then(
        pl.lit("38.78 ± 0.95 C") # Table 3
    ).when(
        (pl.col("sample_name").str.contains("hacat_3h_hsc_20_"))
    ).then(
        pl.lit("40.82 ± 0.67 C") # Table 3
    ).when(
        (pl.col("sample_name").str.contains("hacat_on_hsc_10_"))
    ).then(
        pl.lit("38.87 ± 0.7 C") # Table 3
    ).when(
        (pl.col("sample_name").str.contains("hek3n_3h_hsc_20_"))
    ).then(
        pl.lit("40.8 ± 0.38 C") # Table 1
    ).when(
        (pl.col("sample_name").str.contains("nhek_3h_hsc_20_"))
    ).then(
        pl.lit("40.9 ± 0.2 C") # Table 1
    ).otherwise(
        pl.lit(None)
    ).alias("stress treatment temperature"),

    # stress treatment time
    pl.when(
        (pl.col("sample_name").str.contains("_3h_"))
    ).then(
        pl.lit("180 min.") # 3h.
    ).when(
        (pl.col("sample_name").str.contains("_on_"))
    ).then(
        pl.lit("840 min.") # 14h.
    ).otherwise(
        pl.lit(None)
    ).alias("stress treatment time"),

    # control treatment temperature
    pl.when(
        (pl.col("sample_name").str.contains("hacat_3h_sham_"))
    ).then(
        pl.lit("35.1 ± 0.39 C") # Table 3
    ).when(
        (pl.col("sample_name").str.contains("hacat_on_sham_"))
    ).then(
        pl.lit("35.77 ± 0.23 C") # Table 3
    ).when(
        (pl.col("sample_name").str.contains("hek3n_3h_sham_"))
    ).then(
        pl.lit("35.9 ± 0.2 C") # Table 3
    ).when(
        (pl.col("sample_name").str.contains("nhek_3h_sham_"))
    ).then(
        pl.lit("36.2 ± 0.24 C") # Table 3
    ).otherwise(
        pl.lit(None)
    ).alias("control treatment temperature")
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE234710_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE234710_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE234710_iseq_filtered = GSE234710_iseq[[s.name for s in GSE234710_iseq if not (s.null_count() == GSE234710_iseq.height)]]

GSE234710_combined = GSE234710_PEP.join(
    GSE234710_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE234710_combined = GSE234710_combined[[s.name for s in GSE234710_combined if not (s.null_count() == GSE234710_combined.height)]]

GSE234710_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE234710_modified_metadata.tsv",
    separator='\t'
)

pl.Config.set_tbl_rows(2)
display(GSE234710_combined)

sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,exposure,assembly,sample_accession,rna_library,…,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,…,str,str,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""hacat_3h_hsc_10_r2""","""HaCaT, 3H, HSC 10, R2""","""GSM7473599""","""Public on Jun 20 2024""","""Jun 12 2023""","""Jun 20 2024""","""SRA""",1,"""human keratinocyte cell line (HaCat)""","""Homo sapiens""",9606,"""polyA RNA""","""HaCaT treatement during 3 hours Heat Shock Control equivalent to ""Expo 10 mW/cm²"" Replicate 2, A5""","""GPL20301""","""Frédéric,,Chalmel""","""frederic.chalmel@inserm.fr""","""Inserm U1085-Irset""","""9 avenue du Pr. Léon Bernard""","""Rennes""","""France""",35000,"""France""","""Illumina HiSeq 4000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM7473599""","""NONE""","""GSE234710""",0,"""SRX20661896""","""HaCaT""","""human keratinocyte cell line""","""HSC_10""","""hg38""","""SAMN35715723""","""unknown""",…,"""Transcriptional landscape of human keratinocyte models exposed to 60-GHz millimeter-waves""","""SRA1653935""","""cDNA""","""2024-06-22""",535569628,"""301462729;382033335""","""ILLUMINA""","""missing""","""Generic""","""HaCaT, 3H, HSC 10, R2""","""26ce5bc06057e8391060e59a5209663f;8b4c2f34f08f3c39c036bde19efe80eb""","""SRP442911""",5375492,"""Transcriptional landscape of human keratinocyte models exposed to 60-GHz millimeter-waves""","""HaCaT, 3H, HSC 10, R2""","""Illumina HiSeq 4000""","""Illumina HiSeq 4000 sequencing: GSM7473599: HaCaT, 3H, HSC 10, R2 Homo sapiens RNA-Seq""","""f0a89e6e1164abd9a35e0eab0b8347cb""","""human keratinocyte cell line""",1075098400,"""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM7473599""","""GSM7473599_r1""","""RNA was harvested using RNeasy Micro Kit (Qiagen, Germany) including on-column DNase I treatment according to manufacturer's instructions. Libraries were prepared from 10 ng of total RNA. Poly(A) tail…","""missing""","""GSM7473599_r1""","""HaCaT""","""GSM7473599""","""2024-06-22""","""PRJNA982800""","""SRR24899779""","""SRS17958585""","""2024-06-22""","""GSE234710"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""nhek_3h_sham_0_r3""","""NHEK, 3H, Sham 0, R3""","""GSM7473647""","""Public on Jun 20 2024""","""Jun 12 2023""","""Jun 20 2024""","""SRA""",1,"""human primary culture keratinocyte (NHEK3N)""","""Homo sapiens""",9606,"""polyA RNA""","""NHEK treatement during 3 hours Sham 0 Replicate 3, H1""","""GPL20301""","""Frédéric,,Chalmel""","""frederic.chalmel@inserm.fr""","""Inserm U1085-Irset""","""9 a

In [25]:
defined_pairs_gse234710 = [
    # HaCat cell line (Human keratinocyte cell line)
    # HaCaT cells were exposed either on MMW gradient 5 to 20 mW/cm² during 3 hours or 10 MW/cm² during 14 hours.
    # ("SRR24899791", "SRR24899764"), # hacat_3h_expo_10_r1 vs hacat_3h_sham_0_r1
    # ("SRR24899788", "SRR24899765"), # hacat_3h_expo_10_r2 vs hacat_3h_sham_0_r2
    # ("SRR24899789", "SRR24899766"), # hacat_3h_expo_10_r3 vs hacat_3h_sham_0_r3
    # ("SRR24899790", "SRR24899767"), # hacat_3h_expo_10_r4 vs hacat_3h_sham_0_r4
    # ("SRR24899785", "SRR24899764"), # hacat_3h_expo_20_r1 vs hacat_3h_sham_0_r1
    # ("SRR24899786", "SRR24899765"), # hacat_3h_expo_20_r2 vs hacat_3h_sham_0_r2
    # ("SRR24899787", "SRR24899766"), # hacat_3h_expo_20_r3 vs hacat_3h_sham_0_r3
    # ("SRR24899784", "SRR24899767"), # hacat_3h_expo_20_r4 vs hacat_3h_sham_0_r4
    # ("SRR24899780", "SRR24899764"), # hacat_3h_expo_5_r1 vs hacat_3h_sham_0_r1
    # ("SRR24899781", "SRR24899765"), # hacat_3h_expo_5_r2 vs hacat_3h_sham_0_r2
    # ("SRR24899782", "SRR24899766"), # hacat_3h_expo_5_r3 vs hacat_3h_sham_0_r3
    # ("SRR24899783", "SRR24899767"), # hacat_3h_expo_5_r4 vs hacat_3h_sham_0_r4
    # heat exposed to a temperature equivalent to those reached during the MMW exposure.
    ("SRR24899778", "SRR24899764"), # hacat_3h_hsc_10_r1 vs hacat_3h_sham_0_r1
    ("SRR24899779", "SRR24899765"), # hacat_3h_hsc_10_r2 vs hacat_3h_sham_0_r2
    ("SRR24899776", "SRR24899766"), # hacat_3h_hsc_10_r3 vs hacat_3h_sham_0_r3
    ("SRR24899777", "SRR24899767"), # hacat_3h_hsc_10_r4 vs hacat_3h_sham_0_r4
    ("SRR24899772", "SRR24899764"), # hacat_3h_hsc_20_r1 vs hacat_3h_sham_0_r1
    ("SRR24899773", "SRR24899765"), # hacat_3h_hsc_20_r2 vs hacat_3h_sham_0_r2
    ("SRR24899774", "SRR24899766"), # hacat_3h_hsc_20_r3 vs hacat_3h_sham_0_r3
    ("SRR24899775", "SRR24899767"), # hacat_3h_hsc_20_r4 vs hacat_3h_sham_0_r4
    ("SRR24899768", "SRR24899764"), # hacat_3h_hsc_5_r1 vs hacat_3h_sham_0_r1
    ("SRR24899769", "SRR24899765"), # hacat_3h_hsc_5_r2 vs hacat_3h_sham_0_r2
    ("SRR24899770", "SRR24899766"), # hacat_3h_hsc_5_r3 vs hacat_3h_sham_0_r3
    ("SRR24899771", "SRR24899767"), # hacat_3h_hsc_5_r4 vs hacat_3h_sham_0_r4
    # ("SRR24899762", "SRR24899755"), # hacat_on_expo_10_r1 vs hacat_on_sham_0_r1
    # ("SRR24899760", "SRR24899753"), # hacat_on_expo_10_r2 vs hacat_on_sham_0_r2
    # ("SRR24899763", "SRR24899752"), # hacat_on_expo_10_r3 vs hacat_on_sham_0_r3
    # ("SRR24899761", "SRR24899754"), # hacat_on_expo_10_r4 vs hacat_on_sham_0_r4
    ("SRR24899758", "SRR24899755"), # hacat_on_hsc_10_r1 vs hacat_on_sham_0_r1
    ("SRR24899756", "SRR24899753"), # hacat_on_hsc_10_r2 vs hacat_on_sham_0_r2
    ("SRR24899759", "SRR24899752"), # hacat_on_hsc_10_r3 vs hacat_on_sham_0_r3
    ("SRR24899757", "SRR24899754"), # hacat_on_hsc_10_r4 vs hacat_on_sham_0_r4
    # HEK3N cell line
    # ("SRR24899750", "SRR24899743"), # hek3n_3h_expo_20_r1 vs hek3n_3h_sham_0_r1
    # ("SRR24899748", "SRR24899741"), # hek3n_3h_expo_20_r2 vs hek3n_3h_sham_0_r2
    # ("SRR24899751", "SRR24899740"), # hek3n_3h_expo_20_r3 vs hek3n_3h_sham_0_r3
    # ("SRR24899749", "SRR24899742"), # hek3n_3h_expo_20_r4 vs hek3n_3h_sham_0_r4
    ("SRR24899746", "SRR24899743"), # hek3n_3h_hsc_20_r1 vs hek3n_3h_sham_0_r1
    ("SRR24899744", "SRR24899741"), # hek3n_3h_hsc_20_r2 vs hek3n_3h_sham_0_r2
    ("SRR24899747", "SRR24899740"), # hek3n_3h_hsc_20_r3 vs hek3n_3h_sham_0_r3
    ("SRR24899745", "SRR24899742"), # hek3n_3h_hsc_20_r4 vs hek3n_3h_sham_0_r4
    # NHEK cell line
    # ("SRR24899739", "SRR24899733"), # nhek_3h_expo_20_r1 vs nhek_3h_sham_0_r1
    # ("SRR24899738", "SRR24899732"), # nhek_3h_expo_20_r2 vs nhek_3h_sham_0_r2
    # ("SRR24899737", "SRR24899731"), # nhek_3h_expo_20_r3 vs nhek_3h_sham_0_r3
    ("SRR24899735", "SRR24899733"), # nhek_3h_hsc_20_r1 vs nhek_3h_sham_0_r1
    ("SRR24899734", "SRR24899732"), # nhek_3h_hsc_20_r2 vs nhek_3h_sham_0_r2
    ("SRR24899736", "SRR24899731")  # nhek_3h_hsc_20_r3 vs nhek_3h_sham_0_r3
]

paired_metadata_df_gse234710 = create_paired_metadata(
    combined_df=GSE234710_combined,
    defined_sample_pairs=defined_pairs_gse234710,
    sample_id_column_name="accession"
)
if paired_metadata_df_gse234710 is not None:
    # Strip residual "null vs " prefix and " vs null" suffix from all string columns,
    # which appear when only one side of a pair has a value.
    paired_metadata_df_gse234710 = paired_metadata_df_gse234710.with_columns(
        pl.col(pl.String)
        .str.replace_all(r"^null vs ", "") # control treatment temperature column から "null vs " を削除
        .str.replace_all(r" vs null$", "") # stress treatment temperature column から "vs null" を削除
    )
    pl.Config.set_tbl_rows(2)
    display(paired_metadata_df_gse234710)
    paired_metadata_df_gse234710.write_csv(
        "../../Data/metadata/metadata_paired/GSE234710_paired_metadata.tsv",
        separator='\t'
    )
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,exposure,assembly,…,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR24899778""","""SRR24899764""","""hacat_3h_hsc_10_r1 vs hacat_3h_sham_0_r1""","""HaCaT, 3H, HSC 10, R1 vs HaCaT, 3H, Sham 0, R1""","""GSM7473600 vs GSM7473614""","""Public on Jun 20 2024""","""Jun 12 2023""","""Jun 20 2024""","""SRA""","""1""","""human keratinocyte cell line (HaCat)""","""Homo sapiens""","""9606""","""polyA RNA""","""HaCaT treatement during 3 hours Heat Shock Control equivalent to ""Expo 10 mW/cm²"" Replicate 1, B4 vs HaCaT treatement during 3 hours Sham 0 Replicate 1, F3""","""GPL20301""","""Frédéric,,Chalmel""","""frederic.chalmel@inserm.fr""","""Inserm U1085-Irset""","""9 avenue du Pr. Léon Bernard""","""Rennes""","""France""","""35000""","""France""","""Illumina HiSeq 4000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM7473600 vs GSM7473614""","""NONE""","""GSE234710""","""0""","""SRX20661897 vs SRX20661911""","""HaCaT""","""human keratinocyte cell line""","""HSC_10 vs Sham_0""","""hg38""",…,"""Illumina HiSeq 4000 sequencing: GSM7473600: HaCaT, 3H, HSC 10, R1 Homo sapiens RNA-Seq vs Illumina HiSeq 4000 sequencing: GSM7473614: HaCaT, 3H, Sham 0, R1 Homo sapiens RNA-Seq""","""Transcriptional landscape of human keratinocyte models exposed to 60-GHz millimeter-waves""","""SRA1653935""","""cDNA""","""2024-06-22""","""676269297 vs 96790146""","""379940382;483085382 vs 54950288;68859077""","""ILLUMINA""","""missing""","""Generic""","""HaCaT, 3H, HSC 10, R1 vs HaCaT, 3H, Sham 0, R1""","""34b50ed527439bb016a11e4dd1c054eb;8c5bfe00157203ac9707f46aea26bd95 vs 59ae4619511d5b32b89056071e9abe78;04db82aaf79feb533b5a9186db5dbb91""","""SRP442911""","""6794507 vs 961870""","""Transcriptional landscape of human keratinocyte models exposed to 60-GHz millimeter-waves""","""HaCaT, 3H, HSC 10, R1 vs HaCaT, 3H, Sham 0, R1""","""Illumina HiSeq 4000""","""Illumina HiSeq 4000 sequencing: GSM7473600: HaCaT, 3H, HSC 10, R1 Homo sapiens RNA-Seq vs Illumina HiSeq 4000 sequencing: GSM7473614: HaCaT, 3H, Sham 0, R1 Homo sapiens RNA-Seq""","""208b6b45319b8b9dafd8676cff451151 vs 58c1a9b8d5a10203496cb05199f21708""","""human keratinocyte cell line""","""1358901400 vs 192374000""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM7473600 vs GSM7473614""","""GSM7473600_r1 vs GSM7473614_r1""","""RNA was harvested using RNeasy Micro Kit (Qiagen, Germany) including on-column DNase I treatment according to manufacturer's instructions. Libraries were prepared from 10 ng of total RNA. 

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE255206 (SINGLE) ARTR-seq

- validate date:2025-05-03
- `Dynamics of RNA localization to nuclear speckles are connected to splicing efficiency [ARTR-seq]`
- [Dynamics of RNA localization to nuclear speckles are connected to splicing efficiency](https://doi.org/10.1126/sciadv.adp7727)
- ARTR-seq (reverse transcription–based RNA binding protein binding sites sequencing) という手法のデータ
- ARTR-Seqの論文: [Profiling of RNA-binding protein binding sites by in situ reverse transcription-based sequencing](https://doi.org/10.1038/s41592-023-02146-w)
- GSE255207の論文の別のプロジェクト
- [12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から移動

In [26]:
GSE255206_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE255206/GSE255206_PEP/GSE255206_PEP_raw.csv",
    separator=','
)

display(GSE255206_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,cell_type,genotype,treatment,antibody,assembly,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str
"""hela_mock_control_rep1_artr""",null,null,null,null,null,null,"""Hela_mock_Control_rep1_ARTR""","""GSM8065741""","""Public on Sep 05 2024""","""Feb 06 2024""","""Sep 05 2024""","""SRA""",1,"""Cervical cancer cells""","""Homo sapiens""",9606,"""total RNA""","""GPL24676""","""Jingyi,,Fei""","""jingyifei@uchicago.edu""",7737024370,"""Fei Lab""","""The University of Chicago""","""GCIS W142, 929 E 57th Street""","""Chicago""","""IL""",60637,"""USA""","""Illumina NovaSeq 6000""","""other""","""transcriptomic""","""OTHER""","""GSM8065741""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8065nnn/GSM8065741/suppl/GSM8065741_Hela_mock_Control_rep1.count.txt.gz""","""GSE255206, GSE255208""",0,"""SRX23553091""","""HeLa""","""Cervical cancer cells""","""WT""","""Mock""","""None""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN39846466""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX23553091"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""hepg2_mock_srrm2_rep2_artr""",null,null,null,null,null,null,"""HepG2_mock_SRRM2_rep2_ARTR""","""GSM8065764""","""Public on Sep 05 2024""","""Feb 06 2024""","""Sep 05 2024""","""SRA""",1,"""Liver cancer cells""","""Homo sapiens""",9606,"""total RNA""","""GPL24676""","""Jingyi,,Fei""","""jingyifei@uchicago.edu""",7737024370,"""Fei Lab""","""The University of Chicago""","""GCIS W142, 929 E 57th Street""","""Chicago""","""IL""",60637,"""USA""","""Illumina NovaSeq 6000""","""other""","""transcriptomic""","""OTHER""","""GSM8065764""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8065nnn/GSM8065764/suppl/GSM8065764_HepG2_mock_SRRM2_rep2.count.txt.gz""","""GSE255206, GSE255208""",0,"""SRX23553114""","""HepG2""","""Liver cancer cells""","""WT""","""Mock""","""SRRM2 (Sigma Aldrich, #HPA041411)""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN39846443""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX23553114"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE197992 (SINGLE) TT-chem-seq

<!-- - ✅ DOWNLOAD -->
<!-- - ✅ TRIMMING -->
<!-- - ✅ QUANTIFICATION -->

&nbsp;

- validation date: 2025-05-03
- [An Important Role for RPRD1B in the Heat Shock Response](https://pmc.ncbi.nlm.nih.gov/articles/PMC9583720/)
- おそらく上記の論文由来のデータ
- RPRD1Bの機能を調査しているデータ
- TTchem-seqのデータのプロジェクト
- コントロールデータと熱ストレス曝露のデータが揃っている
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から処理を移動

In [27]:
GSE197992_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE197992/GSE197992_PEP/GSE197992_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column (Illumina)
    pl.lit("https://doi.org/10.1016/j.molcel.2022.01.007").alias("reference_doi") # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE197992_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE197992_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE197992_iseq_filtered = GSE197992_iseq[[s.name for s in GSE197992_iseq if not (s.null_count() == GSE197992_iseq.height)]]

# display(GSE197992_PEP)

In [28]:
GSE197992_combined = GSE197992_PEP.join(
    GSE197992_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE197992_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE197992_ttchem_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE197992_combined)

In [29]:
defined_pairs_gse197992_ttchem_seq = [
    ("SRR18248318", "SRR18248319"), # HS_siNT_B1 [TTchem-seq] (heat-stress) vs NHS_siNT_B1 [TTchem-seq] (control)
    ("SRR18248314", "SRR18248315"), # HS_siNT_B2 [TTchem-seq] (heat-stress) vs NHS_siNT_B2 [TTchem-seq] (control)
    ("SRR18248316", "SRR18248317"), # HS_siRPRD1B_B1 [TTchem-seq] (heat-stress) vs NHS_siRPRD1B_B1 [TTchem-seq] (control)
    ("SRR18248312", "SRR18248313")  # HS_siRPRD1B_B2 [TTchem-seq] (heat-stress) vs NHS_siRPRD1B_B2 [TTchem-seq] (control)
 ]

paired_metadata_df_gse197992_ttchem_seq = create_paired_metadata(
    combined_df=GSE197992_combined,
    defined_sample_pairs=defined_pairs_gse197992_ttchem_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse197992_ttchem_seq is not None:
    paired_metadata_df_gse197992_ttchem_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE197992_ttchem_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse197992_ttchem_seq)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,…,sample_accession,rna_library,reference_doi,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,base_count,library_strategy_right,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR18248318""","""SRR18248319""","""hs_sint_b1_ttchem-seq_ vs nhs_sint_b1_ttchem-seq_""","""null""","""null""","""null""","""null""","""null""","""null""","""HS_siNT_B1 [TTchem-seq] vs NHS_siNT_B1 [TTchem-seq]""","""GSM5935407 vs GSM5935406""","""Public on Sep 08 2022""","""Mar 06 2022""","""Sep 08 2022""","""SRA""","""1""","""MRC5-VA cell line""","""Homo sapiens""","""9606""","""total RNA""","""CUG4410A2, Nascent RNA vs CUG4410A1, Nascent RNA""","""GPL24676""","""Richard,James,Mitter""","""Bioinformatics & Biostatistics""","""The Francis Crick Institute""","""1 Midland Road""","""London""","""LONDON""","""NW1 1AT""","""United Kingdom""","""Illumina NovaSeq 6000""","""other""","""transcriptomic""","""OTHER""","""GSM5935407 vs GSM5935406""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5935nnn/GSM5935407/suppl/GSM5935407_HS_siNT_B1.bigWig vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5935nnn/GSM5935406/suppl/GSM5935406_NHS_siNT_B1.bigWig""","""GSE197992, GSE197995""",…,"""SAMN26490996 vs SAMN26490997""","""unknown""","""https://doi.org/10.1016/j.molcel.2022.01.007""","""SRR18248318 vs SRR18248319""","""Illumina NovaSeq 6000 sequencing: GSM5935407: HS_siNT_B1 [TTchem-seq] Homo sapiens OTHER vs Illumina NovaSeq 6000 sequencing: GSM5935406: NHS_siNT_B1 [TTchem-seq] Homo sapiens OTHER""","""RPRD1B is required for proper heat shock response [TTchem-seq]""","""SRA1381501""","""other""","""2022-09-09""","""2318412091 vs 2192535682""","""3460145458 vs 3162749522""","""ILLUMINA""","""HS_siNT_B1 [TTchem-seq] vs NHS_siNT_B1 [TTchem-seq]""","""7e7ffd8a41a13f2897eda24a36c012c7 vs a56a11a514790bf70101444bc35d2120""","""SRP362753""","""69939984 vs 65358071""","""RPRD1B is required for proper heat shock response [TTchem-seq]""","""HS_siNT_B1 [TTchem-seq] vs NHS_siNT_B1 [TTchem-seq]""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM5935407: HS_siNT_B1 [TTchem-seq] Homo sapiens OTHER vs Illumina NovaSeq 6000 sequencing: GSM5935406: NHS_siNT_B1 [TTchem-seq] Homo sapiens OTHER""","""aa98ad4a5d1ade9988e6f050839e6ff9 vs c7fae1f6ff5b41bdebfb20c26b486588""","""7063938384 vs 6601165171""","""OTHER""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM5935407 vs GSM5935406""","""GSM5935407_r1 vs GSM5935406_r1""","""RNA was extracted by TRIzol (Thermo Fisher Scientific, 15596026) according to the manufacturer's instructions. As a control for equal sample preparation, we spiked-in S. cerevisiae (strain BY4741, MAT…","""GSM5

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE189562 (SINGLE) TT-chem-seq

- validation date 2025-05-03
- [Heat shock induces premature transcript termination and reconfigures the human transcriptome](https://doi.org/10.1016/j.molcel.2022.01.007)
- 以前の研究でも使用していたTTchem-seqのデータ
- 一つのGSM IDに対して2つのSRAのrun IDが付与されている (`srr`カラムが`multi`)
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から処理を移動

In [30]:
GSE189562_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE189562/GSE189562_PEP/GSE189562_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("KAPA RNA HyperPrep kit").alias("rna_library"), # add rna_library column (Roche)
    pl.lit("https://doi.org/10.1016/j.molcel.2022.01.007").alias("reference_doi") # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE189562_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE189562_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE189562_iseq_filtered = GSE189562_iseq[[s.name for s in GSE189562_iseq if not (s.null_count() == GSE189562_iseq.height)]]

# display(GSE189562_PEP)

In [31]:
GSE189562_combined = GSE189562_PEP.join(
    GSE189562_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE189562_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE189562_ttchem_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE189562_combined)

In [32]:
defined_pairs_gse189562_ttchem_seq = [
    ("SRR17038300", "SRR17038298"), # (Run1) CPSF3_wt_HS_TTchem-seq_rep1 vs CPSF3_wt_ctrl_TTchem-seq_rep1
    ("SRR17038301", "SRR17038299"), # (Run2) CPSF3_wt_HS_TTchem-seq_rep1 vs CPSF3_wt_ctrl_TTchem-seq_rep1
    ("SRR17038304", "SRR17038302"), # (Run1) CPSF3_depleted_HS_TTchem-seq_rep1 vs CPSF3_depleted_ctrl_TTchem-seq_rep1
    ("SRR17038305", "SRR17038303"), # (Run2) CPSF3_depleted_HS_TTchem-seq_rep1 vs CPSF3_depleted_ctrl_TTchem-seq_rep1
    ("SRR17038308", "SRR17038306"), # (Run1) CPSF3_wt_HS_TTchem-seq_rep2 vs CPSF3_wt_ctrl_TTchem-seq_rep2
    ("SRR17038309", "SRR17038307"), # (Run2) CPSF3_wt_HS_TTchem-seq_rep2 vs CPSF3_wt_ctrl_TTchem-seq_rep2
    ("SRR17038288", "SRR17038286"), # (Run1) CPSF3_depleted_HS_TTchem-seq_rep2 vs CPSF3_depleted_ctrl_TTchem-seq_rep2
    ("SRR17038289", "SRR17038287"), # (Run2) CPSF3_depleted_HS_TTchem-seq_rep2 vs CPSF3_depleted_ctrl_TTchem-seq_rep2
    ("SRR17038292", "SRR17038290"), # (Run1) CPSF3_wt_HS_TTchem-seq_rep3 vs CPSF3_wt_ctrl_TTchem-seq_rep3
    ("SRR17038293", "SRR17038291"), # (Run2) CPSF3_wt_HS_TTchem-seq_rep3 vs CPSF3_wt_ctrl_TTchem-seq_rep3
    ("SRR17038296", "SRR17038294"), # (Run1) CPSF3_depleted_HS_TTchem-seq_rep3 vs CPSF3_depleted_ctrl_TTchem-seq_rep3
    ("SRR17038297", "SRR17038295")  # (Run2) CPSF3_depleted_HS_TTchem-seq_rep3 vs CPSF3_depleted_ctrl_TTchem-seq_rep3
]

paired_metadata_df_gse189562_ttchem_seq = create_paired_metadata(
    combined_df=GSE189562_combined,
    defined_sample_pairs=defined_pairs_gse189562_ttchem_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse189562_ttchem_seq is not None:
    paired_metadata_df_gse189562_ttchem_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE189562_ttchem_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse189562_ttchem_seq)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,…,rna_library,reference_doi,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,base_count,library_strategy_right,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR17038300""","""SRR17038298""","""cpsf3_wt_hs_ttchem-seq_rep1 vs cpsf3_wt_ctrl_ttchem-seq_rep1""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""multi""","""SRX13228174 vs SRX13228173""","""CPSF3_wt_HS_TTchem-seq_rep1 vs CPSF3_wt_ctrl_TTchem-seq_rep1""","""GSM5703962 vs GSM5703961""","""Public on Jan 24 2022""","""Nov 25 2021""","""Jan 24 2022""","""SRA""","""1""","""MRC5-VA_CPSF3-dTAG-HA""","""Homo sapiens""","""9606""","""total RNA""","""HS.CPSF3_wt.r1 vs ctrl.CPSF3_wt.r1""","""GPL24676""","""Richard,James,Mitter""","""Bioinformatics & Biostatistics""","""The Francis Crick Institute""","""1 Midland Road""","""London""","""LONDON""","""NW1 1AT""","""United Kingdom""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5703962 vs GSM5703961""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5703nnn/GSM5703962/suppl/GSM5703962_HS.CPSF3_wt.r1.bigWig vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5703nnn/GSM5703961/suppl/GSM5703961_ctrl.CPSF3_wt.r1.bi…","""GSE165368, GSE189562""",…,"""KAPA RNA HyperPrep kit""","""https://doi.org/10.1016/j.molcel.2022.01.007""","""SRR17038300 vs SRR17038298""","""Illumina NovaSeq 6000 sequencing: GSM5703962: CPSF3_wt_HS_TTchem-seq_rep1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM5703961: CPSF3_wt_ctrl_TTchem-seq_rep1 Homo sapiens RNA-Seq""","""Heat Shock induces premature transcript termination and reconfigures the human transcriptome""","""SRA1334912""","""cDNA""","""2022-01-26 vs 2022-02-24""","""1085526717 vs 828271238""","""1646780624 vs 1191231011""","""ILLUMINA""","""Generic""","""CPSF3_wt_HS_TTchem-seq_rep1 vs CPSF3_wt_ctrl_TTchem-seq_rep1""","""dfce2ff250c9939e3f00d7fda3f31d8c vs c313783d34a43ab7c36391e3fff820d1""","""SRP347883""","""34420695 vs 26271133""","""Heat Shock induces premature transcript termination and reconfigures the human transcriptome""","""CPSF3_wt_HS_TTchem-seq_rep1 vs CPSF3_wt_ctrl_TTchem-seq_rep1""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM5703962: CPSF3_wt_HS_TTchem-seq_rep1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM5703961: CPSF3_wt_ctrl_TTchem-seq_rep1 Homo sapiens RNA-Seq""","""789aebd690dd87761280bc65763b62f5 vs c7532a3dfaba2699f2edd101acc7721c""","""3476490195 vs 2653384433""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM5703962 vs GSM5703961""","""GSM5703962_r1 vs GSM5703961_r1""","""RNA was extracted by TRIzol (Thermo Fisher Sc

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE165365 (SINGLE) (DREB) TT-chem-seq

- validation date: 2025-05-03
- [Heat shock induces premature transcript termination and reconfigures the human transcriptome](https://doi.org/10.1016/j.molcel.2022.01.007)
- このデータも以前の研究で使用したDRB_TTchem-seqのデータ
- コントロールと熱ストレス曝露のデータが揃っているので念の為収集
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から処理を移動

In [33]:
GSE165365_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE165365/GSE165365_PEP/GSE165365_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("KAPA RNA HyperPrep kit").alias("rna_library"), # add rna_library column (Roche)
    pl.lit("https://doi.org/10.1016/j.molcel.2022.01.007").alias("reference_doi") # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE165365_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE165365_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE165365_iseq_filtered = GSE165365_iseq[[s.name for s in GSE165365_iseq if not (s.null_count() == GSE165365_iseq.height)]]

# display(GSE165365_PEP)

In [34]:
GSE165365_combined = GSE165365_PEP.join(
    GSE165365_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE165365_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE165365_drb_ttchem_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE165365_combined)

In [35]:
defined_pairs_gse165365_ttchem_seq = [
    ("SRR13510285", "SRR13510275"), # hs_drb_ttchem-seq_0min_release_rep1 vs ctrl_drb_ttchem-seq_0min_release_rep1
    ("SRR13510286", "SRR13510276"), # hs_drb_ttchem-seq_10min_release_rep1 vs ctrl_drb_ttchem-seq_10min_release_rep1
    ("SRR13510287", "SRR13510277"), # hs_drb_ttchem-seq_20min_release_rep1 vs ctrl_drb_ttchem-seq_20min_release_rep1
    ("SRR13510288", "SRR13510278"), # hs_drb_ttchem-seq_30min_release_rep1 vs ctrl_drb_ttchem-seq_30min_release_rep1
    ("SRR13510289", "SRR13510279"), # hs_drb_ttchem-seq_40min_release_rep1 vs ctrl_drb_ttchem-seq_40min_release_rep1
    ("SRR13510290", "SRR13510280"), # hs_drb_ttchem-seq_0min_release_rep2 vs ctrl_drb_ttchem-seq_0min_release_rep2
    ("SRR13510291", "SRR13510281"), # hs_drb_ttchem-seq_10min_release_rep2 vs ctrl_drb_ttchem-seq_10min_release_rep2
    ("SRR13510292", "SRR13510282"), # hs_drb_ttchem-seq_20min_release_rep2 vs ctrl_drb_ttchem-seq_20min_release_rep2
    ("SRR13510293", "SRR13510283"), # hs_drb_ttchem-seq_30min_release_rep2 vs ctrl_drb_ttchem-seq_30min_release_rep2
    ("SRR13510294", "SRR13510284"), # hs_drb_ttchem-seq_40min_release_rep2 vs ctrl_drb_ttchem-seq_40min_release_rep2
]

paired_metadata_df_gse165365_ttchem_seq = create_paired_metadata(
    combined_df=GSE165365_combined,
    defined_sample_pairs=defined_pairs_gse165365_ttchem_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse165365_ttchem_seq is not None:
    paired_metadata_df_gse165365_ttchem_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE165365_ttchem_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse165365_ttchem_seq)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,…,rna_library,reference_doi,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR13510285""","""SRR13510275""","""hs_drb_ttchem-seq_0min_release_rep1 vs ctrl_drb_ttchem-seq_0min_release_rep1""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR13510285 vs SRR13510275""","""SRX9921689 vs SRX9921679""","""HS_DRB.TTchem-seq_0min_release_rep1 vs ctrl_DRB.TTchem-seq_0min_release_rep1""","""GSM5032194 vs GSM5032184""","""Public on Jan 24 2022""","""Jan 22 2021""","""Jan 24 2022""","""SRA""","""1""","""MRC5-VA cell line""","""Homo sapiens""","""9606""","""total RNA""","""DRB/TTchem-seq, Nascent RNA""","""GPL20301""","""Richard,James,Mitter""","""Bioinformatics & Biostatistics""","""The Francis Crick Institute""","""1 Midland Road""","""London""","""LONDON""","""NW1 1AT""","""United Kingdom""","""Illumina HiSeq 4000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5032194 vs GSM5032184""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5032nnn/GSM5032194/suppl/GSM5032194_HS_DRB.TTchem-seq_0min_release_rep1.bigWig vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5032nnn/GSM5032184/suppl/GSM503218…","""GSE165365, GSE165368""",…,"""KAPA RNA HyperPrep kit""","""https://doi.org/10.1016/j.molcel.2022.01.007""","""SRR13510285 vs SRR13510275""","""Illumina HiSeq 4000 sequencing: GSM5032194: HS_DRB.TTchem-seq_0min_release_rep1 Homo sapiens RNA-Seq vs Illumina HiSeq 4000 sequencing: GSM5032184: ctrl_DRB.TTchem-seq_0min_release_rep1 Homo sapiens R…","""Heat shock-mediated global gene downregulation is achieved by premature transcription termination [DRB_TTchem-seq]""","""SRA1186909""","""cDNA""","""2022-02-24""","""1773192820 vs 1437969843""","""2390619835 vs 1905337897""","""ILLUMINA""","""Generic""","""HS_DRB.TTchem-seq_0min_release_rep1 vs ctrl_DRB.TTchem-seq_0min_release_rep1""","""bf47ccccecaff8522744a754e8e9cf24 vs 719b9c8a5bd1545054436ebc665fd374""","""SRP302918""","""61620972 vs 50391078""","""Heat shock-mediated global gene downregulation is achieved by premature transcription termination [DRB_TTchem-seq]""","""HS_DRB.TTchem-seq_0min_release_rep1 vs ctrl_DRB.TTchem-seq_0min_release_rep1""","""Illumina HiSeq 4000""","""Illumina HiSeq 4000 sequencing: GSM5032194: HS_DRB.TTchem-seq_0min_release_rep1 Homo sapiens RNA-Seq vs Illumina HiSeq 4000 sequencing: GSM5032184: ctrl_DRB.TTchem-seq_0min_release_rep1 Homo sapiens R…","""8148ecbd1cfef66ebe5efa2fc796ca10 vs fc06e7cf7d870c386f8a4cc6aea1ab09""","""4683193872 vs 3829721928""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM5032194

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE130892 (SINGLE) MeRIP-seq

- validation date: 2025-05-03
- [Limits in the detection of m6A changes using MeRIP/m6A-seq](https://doi.org/10.1038/s41598-020-63355-3)
- 以前の研究で使用していたRNA-seqとMeRIP-seq (methylated RNA immunoprecipitation sequencing) のデータ
- ~~以前の研究では使用していたが、MeRIP-seqの使用方法としてはChIP-seqのような使用方法なので、今回はフィルタリングして収集する~~
- 念の為、MeRIP-seqも収集する
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から処理を移動
- [./12_7_metadata_meripseq.ipynb](./12_7_metadata_meripseq.ipynb)に処理を移動

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE132447 (SINGLE) small RNA-seq

<!-- - ✅ DOWNLOAD -->
<!-- - ✅ TRIMMING -->
<!-- - ✅ QUANTIFICATION -->

&nbsp;

- validation date: 2025-05-03
- [Multiple Alu Exonization in 3′UTR of a Primate-Specific Isoform of CYP20A1 Creates a Potential miRNA Sponge](https://doi.org/10.1093/gbe/evaa233)
-  HIV1-Tat treatment (おそらくtat)のデータをフィルタリングして、RNA-seqとsmall RNA-seqのデータを収集する
- `Heat shock was given at 45°C (±0.2)  for 30 mins in a water bath.`
- small RNA-seqのデータの処理は[./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から移動

In [36]:
GSE132447_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE132447/GSE132447_PEP/GSE132447_PEP_raw.csv",
    separator=','
)

GSE132447_PEP_small_rna_seq = GSE132447_PEP.filter(
    (pl.col("sample_title").str.contains(r'Small_RNA seq_')) & ~(pl.col("sample_title").str.contains(r'_MCF-7'))
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq small RNA library preparation kit").alias("rna_library"), # add rna_library column from GEO webpage (Illumina)
    pl.lit("https://doi.org/10.1093/gbe/evaa233").alias("reference_doi") # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE132447_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE132447_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE132447_iseq_filtered = GSE132447_iseq[[s.name for s in GSE132447_iseq if not (s.null_count() == GSE132447_iseq.height)]]

# display(GSE132447_PEP_small_rna_seq)

In [37]:
GSE132447_combined_small_rna_seq = GSE132447_PEP_small_rna_seq.join(
    GSE132447_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
).filter(
    ~(pl.col("sample_title").str.contains(r'_Tat_')) # filter out HIV1-Tat treatment data
)

GSE132447_combined_small_rna_seq.write_csv(
    "../../Data/metadata/metadata_modified/GSE132447_small_rna_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE132447_combined_small_rna_seq)

In [38]:
# small RNA-seq
defined_pairs_gse132447_small_rna_seq = [
    ("SRR9257637", "SRR9257634"), # Small_RNA seq_Heat_shock_1 hr recovery_Replicate_1 vs Small_RNA seq_Untreated_Replicate_1
    ("SRR9257638", "SRR9257635"), # Small_RNA seq_Heat_shock_1 hr recovery_Replicate_2 vs Small_RNA seq_Untreated_Replicate_2
    ("SRR9257639", "SRR9257636"), # Small_RNA seq_Heat_shock_1 hr recovery_Replicate_3 vs Small_RNA seq_Untreated_Replicate_3
    ("SRR9257640", "SRR9257634"), # Small_RNA seq_Heat_shock_3 hr recovery_Replicate_1 vs Small_RNA seq_Untreated_Replicate_1
    ("SRR9257641", "SRR9257635"), # Small_RNA seq_Heat_shock_3 hr recovery_Replicate_2 vs Small_RNA seq_Untreated_Replicate_2
    ("SRR9257642", "SRR9257636"), # Small_RNA seq_Heat_shock_3 hr recovery_Replicate_3 vs Small_RNA seq_Untreated_Replicate_3
    ("SRR9257643", "SRR9257634"), # Small_RNA seq_Heat_shock_24 hr recovery_Replicate_1 vs Small_RNA seq_Untreated_Replicate_1
    ("SRR9257644", "SRR9257635"), # Small_RNA seq_Heat_shock_24 hr recovery_Replicate_2 vs Small_RNA seq_Untreated_Replicate_2
    ("SRR9257645", "SRR9257636")  # Small_RNA seq_Heat_shock_24 hr recovery_Replicate_3 vs Small_RNA seq_Untreated_Replicate_3
]

paired_metadata_df_gse132447_small_rna_seq = create_paired_metadata(
    combined_df=GSE132447_combined_small_rna_seq,
    defined_sample_pairs=defined_pairs_gse132447_small_rna_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse132447_small_rna_seq is not None:
    paired_metadata_df_gse132447_small_rna_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE132447_small_rna_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse132447_small_rna_seq)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,…,reference_doi,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR9257637""","""SRR9257634""","""small_rna_seq_heat_shock_1_hr_recovery_replicate_1 vs small_rna_seq_untreated_replicate_1""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR9257637 vs SRR9257634""","""SRX6028098 vs SRX6028095""","""Small_RNA seq_Heat_shock_1 hr recovery_Replicate_1 vs Small_RNA seq_Untreated_Replicate_1""","""GSM3864933 vs GSM3864930""","""Public on Jan 21 2021""","""Jun 10 2019""","""Jan 21 2021""","""SRA""","""1""","""Primary neurons""","""Homo sapiens""","""9606""","""total RNA""","""GPL11154""","""vineet,,jha""","""Persistent system""","""LABS""","""Persistent systems""","""9 A/12 Pingale Aryabhatta Persistent Tower,""","""pune""","""Maharashtra""","""411004""","""India""","""Illumina HiSeq 2000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM3864933 vs GSM3864930""","""NONE""","""GSE132447""",…,"""https://doi.org/10.1093/gbe/evaa233""","""SRR9257637 vs SRR9257634""","""Illumina HiSeq 2000 sequencing: GSM3864933: Small_RNA seq_Heat_shock_1 hr recovery_Replicate_1 Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM3864930: Small_RNA seq_Untreated_Replicate_1 H…","""Origin of a novel CYP20A1 lncRNA through 23 Alu exaptations in the human lineage creates a potential multi-miRNA sponge""","""SRA896538""","""cDNA""","""2021-01-23""","""2396467189 vs 2102697391""","""2748619956 vs 2265877289""","""ILLUMINA""","""Generic""","""Small_RNA seq_Heat_shock_1 hr recovery_Replicate_1 vs Small_RNA seq_Untreated_Replicate_1""","""5b2638338383ee7071d21ed13423cbd7 vs 751285d5d711c975b41b60ef45d235c2""","""SRP200969""","""66114467 vs 58507953""","""Origin of a novel CYP20A1 lncRNA through 23 Alu exaptations in the human lineage creates a potential multi-miRNA sponge""","""Small_RNA seq_Heat_shock_1 hr recovery_Replicate_1 vs Small_RNA seq_Untreated_Replicate_1""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM3864933: Small_RNA seq_Heat_shock_1 hr recovery_Replicate_1 Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM3864930: Small_RNA seq_Untreated_Replicate_1 H…","""3a5a84e7688ffd48d6ee2218091e3e56 vs d2f45b5f98f1fe90a9344da7bcc7b152""","""Primary neurons""","""3371837817 vs 2983905603""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM3864933 vs GSM3864930""","""GSM3864933_r1 vs GSM3864930_r1""","""RNA was isolated from monolayer cells using TRIzol, following manufacturer's protocol. TruSeq Stranded Total RNA Sample Preparation Guide was fo

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE247043 (PAIRED) single-cell RNA-seq

- validation date: 2025-05-03
- [Pre-print: Spatiotemporal Single-Cell Rna Sequencing Reveals that Cd4+T Subpopulation with High Metabolic Flux Plays Pivotal Roles Under in Hyperthermia Treatment Warts](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4636648)
- これはsingle-cell RNA-seq (scRNA-seq) のデータ
- ~~bulk RNA-seqと同様に解析することはできなくはないが、2ペアという数の少なさなので、今回は解析に使用しない~~
- 念の為収集
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から処理を移動

In [39]:
GSE247043_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE247043/GSE247043_PEP/GSE247043_PEP_raw.csv",
    separator=','
)

display(GSE247043_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,sample_supplementary_file_3,sample_series_id,sample_data_row_count,big_key,tissue,cell_type,treatment,assembly,supplementary_files_format_and_content,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str
"""ca1_control_scrnaseq""",null,null,null,null,null,null,"""CA1_Control_scRNAseq""","""GSM7882544""","""Public on Nov 30 2024""","""Nov 06 2023""","""Nov 30 2024""","""SRA""",1,"""Condyloma acuminatum""","""Homo sapiens""",9606,"""total RNA""","""10×Genomics""","""GPL24676""","""Xueli,,Niu""","""niu113123@Gmail.com""","""China Medical University""","""No.77 Puhe Road, Shenyang North New Area""","""Shenyang""",110122,"""China""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic single cell""","""RNA-Seq""","""GSM7882544""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM7882nnn/GSM7882544/suppl/GSM7882544_A2_37barcodes.tsv.gz""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM7882nnn/GSM7882544/suppl/GSM7882544_A2_37genes.tsv.gz""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM7882nnn/GSM7882544/suppl/GSM7882544_A2_37matrix.mtx.gz""","""GSE247043""",0,"""SRX22378428""","""Condyloma acuminatum""","""T cells""","""37 degreeC water bath,30min""","""mm10""","""Tab-separted value files and matrix files""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN38113603""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX22378428"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ca2_hyperthermia_scrnaseq""",null,null,null,null,null,null,"""CA2_Hyperthermia_scRNAseq""","""GSM7882547""","""Public on Nov 30 2024""","""Nov 06 2023""","""Nov 30 2024""","""SRA""",1,"""Condyloma acuminatum""","""Homo sapiens""",9606,"""total RNA""","""10×Genomics""","""GPL24676""","""Xueli,,Niu""","""niu113123@Gmail.com""","""China Medical University""","""No.77 Puhe Road, Shenyang North New Area""","""Shenyang""",110122,"""China""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic single cell""","""RNA-Seq""","""GSM7882547""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM7882nnn/GSM7882547/suppl/GSM7882547_CA3_44barcodes.tsv.gz""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM7882nnn/GSM7882547/suppl/GSM7882547_CA3_44genes.tsv.gz""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM7882nnn/GSM7882547/suppl/GSM7882547_CA3_44matrix.mtx.gz""","""GSE247043""",0,"""SRX22378431""","""Condyloma acuminatum""","""T cells""","""44 degreeC water bath,30min""","""mm10""","""Tab-separted value files and matrix files""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN38113600""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX22378431"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE123980 (PAIRED) TT-seq, mNET-seq

- validate date: 2025-05-04
- [The pause-initiation limit restricts transcription activation in human cells](https://doi.org/10.1038/s41467-019-11536-8)
- TT-seq,RNA-seq,mNET-seqなどの複数の手法が含まれている
- [TT-seqに関する論文: TT-seq maps the human transient transcriptome](https://doi.org/10.1126/science.aad9841)
- [mNET-seq](https://jp.illumina.com/science/sequencing-method-explorer/kits-and-arrays/mnet-seq.html)
- 一応念の為全てのデータを収集
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から処理を移動

In [40]:
ID = "GSE123980"

GSE123980_PEP = pl.read_csv(
    f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("Ovation Universal RNA-Seq System").alias("rna_library"), # add rna_library column from GEO webpage (NuGEN)
    pl.lit("https://doi.org/10.1038/s41467-019-11536-8").alias("reference_doi") # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

GSE123980_PEP_tt_seq = GSE123980_PEP.filter(
    pl.col("sample_title").str.contains(r'TT-seq_')
)

GSE123980_PEP_mnet_seq = GSE123980_PEP.filter(
    pl.col("sample_title").str.contains(r'mNET-seq_')
)

# metadata from iseq
GSE123980_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE123980_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE123980_iseq_filtered = GSE123980_iseq[[s.name for s in GSE123980_iseq if not (s.null_count() == GSE123980_iseq.height)]]

# display(GSE123980_PEP_tt_seq)
# display(GSE123980_PEP_mnet_seq)

In [41]:
GSE123980_tt_seq_combined = GSE123980_PEP_tt_seq.join(
    GSE123980_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE123980_tt_seq_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE123980_tt_seq_modified_metadata.tsv",
    separator='\t'
)

GSE123980_mnet_seq_combined = GSE123980_PEP_mnet_seq.join(
    GSE123980_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE123980_mnet_seq_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE123980_mnet_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE123980_tt_seq_combined)
# display(GSE123980_mnet_seq_combined)

In [42]:
# TT-seq
defined_pairs_gse123980_tt_seq = [
    ("SRR8334015", "SRR8334013"), # TT-seq_K562_HS15_rep1 vs TT-seq_K562_Ctrl_rep1
    ("SRR8334016", "SRR8334014"), # TT-seq_K562_HS15_rep2 vs TT-seq_K562_Ctrl_rep2
    ("SRR8334017", "SRR8334013"), # TT-seq_K562_HS30__rep1 vs TT-seq_K562_Ctrl_rep1
    ("SRR8334018", "SRR8334014"), # TT-seq_K562_HS30_rep2 vs TT-seq_K562_Ctrl_rep2
    ("SRR8334035", "SRR8334031"), # TT-seq_Raji-CDK9as_Solvent_HS30_rep1 vs TT-seq_Raji-CDK9as_Solvent_rep1
    ("SRR8334036", "SRR8334032"), # TT-seq_Raji-CDK9as_Solvent_HS30_rep2 vs TT-seq_Raji-CDK9as_Solvent_rep2
    ("SRR8334037", "SRR8334033"), # TT-seq_Raji-CDK9as_CDK9inhibited_HS30_rep1 vs TT-seq_Raji-CDK9as_CDK9inhibited_rep1
    ("SRR8334038", "SRR8334034")  # TT-seq_Raji-CDK9as_CDK9inhibited_HS30_rep2 vs TT-seq_Raji-CDK9as_CDK9inhibited_rep2
]

paired_metadata_df_gse123980_tt_seq = create_paired_metadata(
    combined_df=GSE123980_tt_seq_combined,
    defined_sample_pairs=defined_pairs_gse123980_tt_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse123980_tt_seq is not None:
    paired_metadata_df_gse123980_tt_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE123980_tt_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse123980_tt_seq)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,gse_id,…,rna_library,reference_doi,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR8334015""","""SRR8334013""","""tt-seq_k562_hs15_rep1 vs tt-seq_k562_ctrl_rep1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR8334015 vs SRR8334013""","""SRX5145533 vs SRX5145531""","""TT-seq_K562_HS15_rep1 vs TT-seq_K562_Ctrl_rep1""","""GSM3518107 vs GSM3518105""","""Public on Jul 10 2019""","""Dec 17 2018""","""Jul 10 2019""","""SRA""","""1""","""K562 erythroleukemia cells""","""Homo sapiens""","""9606""","""total RNA""","""Ovation Universal RNA-Seq System (NuGEN), TT-seq, RNA-seq (Schwalb et al., 2016), or mNET-seq (Nojima et al., 2016; Schlackow et al., 2017).""","""GPL18573""","""Björn,,Schwalb""","""bschwal@gwdg.de""","""MPI""","""Fassberg 11""","""Göttingen""","""37077""","""Germany""","""Illumina NextSeq 500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM3518107 vs GSM3518105""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3518nnn/GSM3518107/suppl/GSM3518107_L_with_1_AAGAGG_3_K562_HS15.minus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3518nnn/GSM3518105/suppl/GSM3518105_L_wi…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3518nnn/GSM3518107/suppl/GSM3518107_L_with_1_AAGAGG_3_K562_HS15.plus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3518nnn/GSM3518105/suppl/GSM3518105_L_wit…","""GSE123980""",…,"""Ovation Universal RNA-Seq System""","""https://doi.org/10.1038/s41467-019-11536-8""","""SRR8334015 vs SRR8334013""","""NextSeq 500 sequencing: GSM3518107: TT-seq_K562_HS15_rep1 Homo sapiens RNA-Seq vs NextSeq 500 sequencing: GSM3518105: TT-seq_K562_Ctrl_rep1 Homo sapiens RNA-Seq""","""The pause-initiation limit restricts transcription activation in human cells""","""SRA824487""","""cDNA""","""2019-07-11""","""10046244804 vs 9383531588""","""7261173774;7430552039 vs 6766405444;6939916984""","""ILLUMINA""","""Generic""","""TT-seq_K562_HS15_rep1 vs TT-seq_K562_Ctrl_rep1""","""1c7fb89123adca1ed620347186ef0cd4;c820e3be42c4c8865374cf9ec99f9a26 vs d12b9335454be9eb281418d4ffefa54d;627bd970f46c3667741fee97fd21d18e""","""SRP173667""","""161455492 vs 149673334""","""The pause-initiation limit restricts transcription activation in human cells""","""TT-seq_K562_HS15_rep1 vs TT-seq_K562_Ctrl_rep1""","""NextSeq 500""","""NextSeq 500 sequencing: GSM3518107: TT-seq_K562_HS15_rep1 Homo sapiens RNA-Seq vs NextSeq 500 sequencing: GSM3518105: TT-seq_K562_Ctrl_rep1 Homo sapiens RNA-Seq""","""a83b35300ac54cb2a8eaa9c00f933472 vs a8eee182b907df64159dfb91ccc5a49c""","""24435140753 vs 22658102303""","""RNA

In [43]:
# mNET-seq
defined_pairs_gse123980_mnet_seq = [
    ("SRR8334027", "SRR8334025"), # mNET-seq_K562_HS15_rep1 vs mNET-seq_K562_Ctrl_rep1
    ("SRR8334028", "SRR8334026"), # mNET-seq_K562_HS15_rep2 vs mNET-seq_K562_Ctrl_rep2
    ("SRR8334029", "SRR8334025"), # mNET-seq_K562_HS30__rep1 vs mNET-seq_K562_Ctrl_rep1
    ("SRR8334030", "SRR8334026")  # mNET-seq_K562_HS30_rep2 vs mNET-seq_K562_Ctrl_rep2
]

paired_metadata_df_gse123980_mnet_seq = create_paired_metadata(
    combined_df=GSE123980_mnet_seq_combined,
    defined_sample_pairs=defined_pairs_gse123980_mnet_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse123980_mnet_seq is not None:
    paired_metadata_df_gse123980_mnet_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE123980_mnet_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse123980_mnet_seq)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,gse_id,…,rna_library,reference_doi,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR8334027""","""SRR8334025""","""mnet-seq_k562_hs15_rep1 vs mnet-seq_k562_ctrl_rep1""","""size fractionation""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR8334027 vs SRR8334025""","""SRX5145545 vs SRX5145543""","""mNET-seq_K562_HS15_rep1 vs mNET-seq_K562_Ctrl_rep1""","""GSM3518119 vs GSM3518117""","""Public on Jul 10 2019""","""Dec 17 2018""","""Jul 10 2019""","""SRA""","""1""","""K562 erythroleukemia cells""","""Homo sapiens""","""9606""","""total RNA""","""TruSeq Small RNA Library (Illumina), TT-seq, RNA-seq (Schwalb et al., 2016), or mNET-seq (Nojima et al., 2016; Schlackow et al., 2017).""","""GPL18573""","""Björn,,Schwalb""","""bschwal@gwdg.de""","""MPI""","""Fassberg 11""","""Göttingen""","""37077""","""Germany""","""Illumina NextSeq 500""","""size fractionation""","""transcriptomic""","""ncRNA-Seq""","""GSM3518119 vs GSM3518117""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3518nnn/GSM3518119/suppl/GSM3518119_N_tCTD_1_GACGAC_1_K562_HS15.minus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3518nnn/GSM3518117/suppl/GSM3518117_N_tC…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3518nnn/GSM3518119/suppl/GSM3518119_N_tCTD_1_GACGAC_1_K562_HS15.plus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3518nnn/GSM3518117/suppl/GSM3518117_N_tCT…","""GSE123980""",…,"""Ovation Universal RNA-Seq System""","""https://doi.org/10.1038/s41467-019-11536-8""","""SRR8334027 vs SRR8334025""","""NextSeq 500 sequencing: GSM3518119: mNET-seq_K562_HS15_rep1 Homo sapiens ncRNA-Seq vs NextSeq 500 sequencing: GSM3518117: mNET-seq_K562_Ctrl_rep1 Homo sapiens ncRNA-Seq""","""The pause-initiation limit restricts transcription activation in human cells""","""SRA824487""","""size fractionation""","""2019-07-11""","""11823218562 vs 10812566404""","""6738791246;6686724495 vs 6243425515;6307729313""","""ILLUMINA""","""Generic""","""mNET-seq_K562_HS15_rep1 vs mNET-seq_K562_Ctrl_rep1""","""c4eb607f5612cbae22615e44100a1954;c819597ffbcaa2681302e6534380d29f vs 9a9946d34539a452b7d710f716bb5e13;f4971df3aa8ae19614047af9b6d7984d""","""SRP173667""","""185761345 vs 169024048""","""The pause-initiation limit restricts transcription activation in human cells""","""mNET-seq_K562_HS15_rep1 vs mNET-seq_K562_Ctrl_rep1""","""NextSeq 500""","""NextSeq 500 sequencing: GSM3518119: mNET-seq_K562_HS15_rep1 Homo sapiens ncRNA-Seq vs NextSeq 500 sequencing: GSM3518117: mNET-seq_K562_Ctrl_rep1 Homo sapiens ncRNA-Seq""","""b7d8fb01918dbb0c565ee0f93cce5261 vs ae0

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE128365 (PAIRED) scEdU-seq

- validate date: 2025-05-04
- [Quantifying DNA replication speeds in single cells by scEdU-seq](https://doi.org/10.1038/s41592-024-02308-4)
- `single-cell 5-ethynyl-2′-deoxyuridine sequencing (scEdU-seq)` という手法に関するデータ
- nacsent RNAの検出ができる手法
- ~~一応 scRNA-seqの亜種なので、今回は収集しない (もしかしたら後ほど収集するかも？)~~
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から処理を移動

In [44]:
ID = "GSE128365"

GSE128365_PEP = pl.read_csv(
    f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("organism") == "Homo sapiens") &
    (pl.col("sample_source_name_ch1").str.contains(r'HeatShock-'))
)

# pl.Config.set_fmt_str_lengths(200)
display(GSE128365_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_type,genetic_tag,library_strategy,supplementary_files_format_and_content,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str
"""heatshock-37c-tot-dmso-45min-k562-plate1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""multi""","""SRX7815439""","""HeatShock-37C-Tot-DMSO-45min-K562-plate1""","""GSM4364034""","""Public on Mar 05 2020""","""Feb 27 2020""","""Mar 05 2020""","""SRA""",1,"""HeatShock-37C-Tot-DMSO-45min-K562""","""Homo sapiens""",9606,"""Cells were incubated with the EU and/or DMSO for the desired time (e.g. 60 min). The final concentration of EU was 200 µM for RPE1-FUCCI chase experiments, 400 µM for RPE1-FUCCI or K562 pulse experime…","""RPE1-FUCCI cells were cultured in Dulbecco's modified Eagle's medium (DMEM)/F12, supplemented with GlutaMAX (Gibco), FBS (Gibco), and penicillin/streptomycin (Gibco), following standard procedures. K5…","""polyA RNA""","""After incubation with EU, cells were dissociated, fixed with paraformaldehyde, and azide-biotin was clicked in situ after cell permeabilization to mark EU-labeled RNA. We then sorted single cells in 3…",null,"""Read 2 of Sequencing liberties were using STAR 2.5 with default parameters. If the length of the poly-T track in read 1 was less than 19 bases, the read was discarded., The number of UMI count per gen…","""GPL18573""","""Nico,,Battich""","""nico.battich@googlemail.com""","""Hubrecht Institute""","""Uppsalalaan 8""","""Utrecht""","""Utrecht""","""3584 CT""","""Netherlands""","""Illumina NextSeq 500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM4364034""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4364nnn/GSM4364034/suppl/GSM4364034_SupplementaryData_HeatShock_37C-unlabeled-DMSO-45min-K562-plate1_umiCount_spliced.csv.gz""","""GSE128365""",0,"""SRX7815439""",null,null,null,"""The number of UMI count per gene""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN14237934""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX7815439"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""heatshock-42c-tot-eu-45min-k562-plate4""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""multi""","""SRX7815444""","""HeatShock-42C-Tot-EU-45min-K562-plate4""","""GSM4364039""","""Public on Mar 05 2020""","""Feb 27 2020""","""Mar 05 2020""","""SRA""",1,"""HeatShock-42C-Tot-EU-45min-K562""","""Homo sapiens""",9606,"""Cells were incubated with the EU and/or DMSO for the desired time (e.g. 60 min). The final concentration of EU was 200 µM for RPE1-FUCCI chase experiments, 400 µM for RPE1-FUCCI or K562 pulse experime…","""RPE1-FUCCI cells were cultured in Dulbecco's modified Eagle's medium (DMEM)/F12, supplemented with GlutaMAX (Gibco), FBS (Gibco), and penicillin/streptomycin (Gibco), following standard procedures. K5…","""polyA RNA""","""After incubation with EU, cells were dissociated, fixed with paraformaldehyde, and azide-biotin was clicked in situ after cell permeabilization to mark EU-labeled RNA. We then sorted single cells in 3…",null,"""Read 2 of Sequencing liberties were using STAR 2.5 with default p

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE100469 (PAIRED) 4sU-seq

- validate date: 2025-05-04
- [HSV-1-induced disruption of transcription termination resembles a cellular stress response but selectively increases chromatin accessibility downstream of genes](https://doi.org/10.1371/journal.ppat.1006954)
- 4sU-seqなので、新生RNAを検出しているデータ
- 念の為収集
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から処理を移動

In [45]:
ID = "GSE100469"

GSE100469_PEP = pl.read_csv(
    f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
    separator=','
).filter(
    ~pl.col("sample_title").str.contains(r' Salt ')
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq Stranded Total RNA kit").alias("rna_library"), # add rna_library column from GEO webpage (Illumina)
    pl.lit("https://doi.org/10.1371/journal.ppat.1006954").alias("reference_doi") # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

GSE100469_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE100469_metadata.tsv",
    separator='\t'
)
GSE100469_iseq_filtered = GSE100469_iseq[[s.name for s in GSE100469_iseq if not (s.null_count() == GSE100469_iseq.height)]]

# display(GSE100469_PEP)

In [46]:
GSE100469_combined = GSE100469_PEP.join(
    GSE100469_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE100469_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE100469_modified_metadata.tsv",
    separator='\t'
)

# display(GSE100469_combined)

In [47]:
defined_pairs_gse100469_4sU_seq = [
    ("SRR5756320", "SRR5756304"), # 0-1 h Heat stress 4sU-RNA - Rep 1 vs Mock 1 4sU-RNA - Rep 1
    ("SRR5756321", "SRR5756305"), # 0-1 h Heat stress 4sU-RNA - Rep 1 vs Mock 1 4sU-RNA - Rep 1
    ("SRR5756322", "SRR5756306"), # 0-1 h Heat stress 4sU-RNA - Rep 1 vs Mock 1 4sU-RNA - Rep 1
    ("SRR5756323", "SRR5756307"), # 0-1 h Heat stress 4sU-RNA - Rep 1 vs Mock 1 4sU-RNA - Rep 1
    ("SRR5756324", "SRR5756308"), # 1-2 h Heat stress 4sU-RNA - Rep 1 vs Mock 2 4sU-RNA - Rep 1
    ("SRR5756325", "SRR5756309"), # 1-2 h Heat stress 4sU-RNA - Rep 1 vs Mock 2 4sU-RNA - Rep 1
    ("SRR5756326", "SRR5756310"), # 1-2 h Heat stress 4sU-RNA - Rep 1 vs Mock 2 4sU-RNA - Rep 1
    ("SRR5756327", "SRR5756311"), # 1-2 h Heat stress 4sU-RNA - Rep 1 vs Mock 2 4sU-RNA - Rep 1
    ("SRR5756344", "SRR5756328"), # 0-1 h Heat stress 4sU-RNA -  Rep 2 vs Mock 1 4sU-RNA - Rep 2
    ("SRR5756345", "SRR5756329"), # 0-1 h Heat stress 4sU-RNA -  Rep 2 vs Mock 1 4sU-RNA - Rep 2
    ("SRR5756346", "SRR5756330"), # 0-1 h Heat stress 4sU-RNA -  Rep 2 vs Mock 1 4sU-RNA - Rep 2
    ("SRR5756347", "SRR5756331"), # 0-1 h Heat stress 4sU-RNA -  Rep 2 vs Mock 1 4sU-RNA - Rep 2
    ("SRR5756348", "SRR5756332"), # 1-2 h Heat stress 4sU-RNA -  Rep 2 vs Mock 2 4sU-RNA - Rep 2
    ("SRR5756349", "SRR5756333"), # 1-2 h Heat stress 4sU-RNA -  Rep 2 vs Mock 2 4sU-RNA - Rep 2
    ("SRR5756350", "SRR5756334"), # 1-2 h Heat stress 4sU-RNA -  Rep 2 vs Mock 2 4sU-RNA - Rep 2
    ("SRR5756351", "SRR5756335")  # 1-2 h Heat stress 4sU-RNA -  Rep 2 vs Mock 2 4sU-RNA - Rep 2
]

paired_metadata_df_gse100469_4sU_seq = create_paired_metadata(
    combined_df=GSE100469_combined,
    defined_sample_pairs=defined_pairs_gse100469_4sU_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse100469_4sU_seq is not None:
    paired_metadata_df_gse100469_4sU_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE100469_4sU_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse100469_4sU_seq)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,…,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,sra_aspera,fastq_galaxy,ncbi_reporting_standard,sample_description_right,sra_galaxy,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,cell_type_right,fastq_ftp,base_count,library_strategy,library_source,sra_ftp,library_layout,status,fastq_aspera,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR5756320""","""SRR5756304""","""0-1_h_heat_stress_4su-rna_-_rep_1 vs mock_1_4su-rna_-_rep_1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""multi""","""SRX2956488 vs SRX2956484""","""0-1 h Heat stress 4sU-RNA - Rep 1 vs Mock 1 4sU-RNA - Rep 1""","""GSM2684050 vs GSM2684046""","""Public on Mar 08 2018""","""Jun 26 2017""","""May 15 2019""","""SRA""","""1""","""0-1 h Heat stress 4sU-RNA vs Mock 1 4sU-RNA""","""Homo sapiens""","""9606""","""total RNA""","""heat_1h_1, Purified from total RNA using 4sU-tagging vs mock1_1, Purified from total RNA using 4sU-tagging""","""GPL18573""","""Caroline,,Friedel""","""Institut für Informatik""","""Ludwig-Maximilians-Universität München""","""Amalienstr. 17""","""München""","""80333""","""Germany""","""Illumina NextSeq 500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM2684050 vs GSM2684046""","""NONE""","""GSE100469""","""0""",…,"""cDNA""","""2018-03-10""","""679889097 vs 635172479""","""473598910;489847187 vs 430773876;448924387""","""ILLUMINA""","""fasp.sra.ebi.ac.uk:/vol1/srr/SRR575/000/SRR5756320 vs fasp.sra.ebi.ac.uk:/vol1/srr/SRR575/004/SRR5756304""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR575/000/SRR5756320/SRR5756320_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR575/000/SRR5756320/SRR5756320_2.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR575/004/SRR5756304/…","""Generic""","""0-1 h Heat stress 4sU-RNA - Rep 1 vs Mock 1 4sU-RNA - Rep 1""","""ftp.sra.ebi.ac.uk/vol1/srr/SRR575/000/SRR5756320 vs ftp.sra.ebi.ac.uk/vol1/srr/SRR575/004/SRR5756304""","""70fbd1b7916632116525c1ef7041d152;3a2807da058e38b574a75d9363a46327 vs 58fa81112c1b7da972a37ff0b6166043;ce71eac50ca9d216f886af9d5d89338f""","""SRP110507""","""11200505 vs 10423543""","""4sU-seq of HFF exposed to salt and heat stress""","""0-1 h Heat stress 4sU-RNA - Rep 1 vs Mock 1 4sU-RNA - Rep 1""","""NextSeq 500""","""NextSeq 500 sequencing: GSM2684050: 0-1 h Heat stress 4sU-RNA - Rep 1 Homo sapiens RNA-Seq vs NextSeq 500 sequencing: GSM2684046: Mock 1 4sU-RNA - Rep 1 Homo sapiens RNA-Seq""","""c32461e1578b21b75be9b6eb08ed32fd vs c2adb5806b7ac8cebce195f79d83a9a7""","""human foreskin fibroblasts""","""ftp.sra.ebi.ac.uk/vol1/fastq/SRR575/000/SRR5756320/SRR5756320_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR575/000/SRR5756320/SRR5756320_2.fastq.gz vs ftp.sra.ebi.ac.uk/vol1/fastq/SRR575/004/SRR5756304/…","""1685382528 vs 1570075159""","""RNA-Seq""","""TRANSCRIPTOMIC""","""ftp.sra.ebi.ac.uk/vol1/srr/SRR575/000/SRR5756320 vs ftp.sra.ebi.ac.uk/vol1/sr

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE127890 (SINGLE) Nanopore (nano-ID)

- validate date 2025-05-04
- [Native molecule sequencing by nano-ID reveals synthesis and stability of RNA isoforms](https://doi.org/10.1101/gr.257857.119)
- 以前の研究でも使用しているnanoporeのデータで、RNAのisoformを検出するための手法 nano-ID (nanopore sequencing–based isoform dynamics) を開発した論文
- `To show the advantages of nano-ID, we subjected human K562 cells to heat shock (42°C) for 60 min in the presence of 5EU (5EU 60 min HS) (Fig. 4A).`
- 上記のような記述に関するデータを収集
- [./12_4_metadata_modified_false_no_control.ipynb](./12_4_metadata_modified_false_no_control.ipynb)から処理を移動

In [48]:
GSE127890_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE127890/GSE127890_PEP/GSE127890_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_organism_ch1") == "Homo sapiens") & # remove synthetic construct sample data
    (pl.col("sample_title").str.contains(r'5EU 60 min')) &
    ~(pl.col("sample_title").str.contains(r'5EU 60 min replicate 6')) # No replicate 6 exposed to heat shock, so this data was removed
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("SQK-RNA001").alias("rna_library"), # add rna_library column from GEO webpage (Oxford Nanopore Technologies)
    pl.lit("https://doi.org/10.1101/gr.257857.119").alias("reference_doi") # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE127890 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE127890_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE127890_iseq_filtered = GSE127890[[s.name for s in GSE127890 if not (s.null_count() == GSE127890.height)]]

# display(GSE127890_PEP)

In [49]:
GSE127890_combined = GSE127890_PEP.join(
    GSE127890_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE127890_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE127890_modified_metadata.tsv",
    separator='\t'
)

# display(GSE127890_combined)

In [50]:
defined_pairs_gse127890 = [
    ("SRR10953094", "SRR10953082"), # 5EU 60 min HS replicate 1 vs 5EU 60 min replicate 1
    ("SRR10953095", "SRR10953083"), # 5EU 60 min HS replicate 2 vs 5EU 60 min replicate 2
    ("SRR10953096", "SRR10953084"), # 5EU 60 min HS replicate 3 vs 5EU 60 min replicate 3
    ("SRR10953097", "SRR10953085"), # 5EU 60 min HS replicate 4 vs 5EU 60 min replicate 4
    ("SRR10953098", "SRR10953086")  # 5EU 60 min HS replicate 5 vs 5EU 60 min replicate 5
]

paired_metadata_df_gse127890 = create_paired_metadata(
    combined_df=GSE127890_combined, 
    defined_sample_pairs=defined_pairs_gse127890,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse127890 is not None:
    paired_metadata_df_gse127890.write_csv(
        "../../Data/metadata/metadata_paired/GSE127890_paired_metadata.tsv",
        separator='\t'
    )

# display(paired_metadata_df_gse127890)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE180253 (SINGLE) m6A-seq

- validation date: 2025-05-06
- [Inhibition of HSP90 as a Strategy to Radiosensitize Glioblastoma: Targeting the DNA Damage Response and Beyond](https://doi.org/10.3389/fonc.2021.612354)
- m6A-seqのデータが含まれている...
- [MeRIP-Seq/m6A-seq](https://jp.illumina.com/science/sequencing-method-explorer/kits-and-arrays/merip-seq-m6a-seq.html)
- `heat_shock_` と `Untreated_` のデータを念の為収集する
- m6A-seqのデータ処理を[./12_4_metadata_modified_false_no_control.ipynb](./12_4_metadata_modified_false_no_control.ipynb)から移動

In [51]:
GSE180253_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE180253/GSE180253_PEP/GSE180253_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("protocol") == "cDNA") & # remove CAGE-seq data
    ((pl.col("sample_title").str.contains(r'heat_shock_')) | 
     (pl.col("sample_title").str.contains(r'Untreated_'))) & # remove HSF1 knockdown data
    ~(pl.col("sample_title").str.contains(r'Hypoxia_')) # remove HSF1 knockdown data
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column from GEO page
    pl.lit("https://doi.org/10.3389/fonc.2021.612354").alias("reference_doi") # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

GSE180253_PEP_m6am_seq = GSE180253_PEP.filter(
    ~(pl.col("sample_description") == "RNA-seq")
)

# metadata from iseq
GSE180253 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE180253_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE180253_iseq_filtered = GSE180253[[s.name for s in GSE180253 if not (s.null_count() == GSE180253.height)]]


# display(GSE180253_PEP)

In [52]:
GSE180253_combined_m6am_seq = GSE180253_PEP_m6am_seq.join(
    GSE180253_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE180253_combined_m6am_seq.write_csv(
    "../../Data/metadata/metadata_modified/GSE180253_m6am_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE180253_combined_m6am_seq)

In [53]:
# m6Am-seq
defined_pairs_gse180253_m6am_seq = [
    ("SRR15177493", "SRR15177507"), # heat_shock_FTO_m6AIP-rep1 vs Untreated_FTO_m6AIP-rep1
    ("SRR15177494", "SRR15177508"), # heat_shock_FTO_m6AIP-rep2 vs Untreated_FTO_m6AIP-rep2
    ("SRR15177496", "SRR15177510"), # heat_shock_nonFTO_m6AIP-rep1 vs Untreated_nonFTO_m6AIP-rep1
    ("SRR15177497", "SRR15177511"), # heat_shock_nonFTO_m6AIP-rep2 vs Untreated_nonFTO_m6AIP-rep2
    ("SRR15177498", "SRR15177512"), # heat_shock_nonFTO_m7GIP-rep1 vs Untreated_nonFTO_m7GIP-rep1
    ("SRR15177499", "SRR15177513")  # heat_shock_nonFTO_m7GIP-rep2 vs Untreated_nonFTO_m7GIP-rep2
]

paired_metadata_df_gse180253_m6am_seq = create_paired_metadata(
    combined_df=GSE180253_combined_m6am_seq, 
    defined_sample_pairs=defined_pairs_gse180253_m6am_seq,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse180253_m6am_seq is not None:
    paired_metadata_df_gse180253_m6am_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE180253_m6am_seq_paired_metadata.tsv",
        separator='\t'
    )

display(paired_metadata_df_gse180253_m6am_seq)

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,…,reference_doi,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR15177493""","""SRR15177507""","""heat_shock_fto_m6aip-rep1 vs untreated_fto_m6aip-rep1""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR15177493 vs SRR15177507""","""SRX11484751 vs SRX11484765""","""heat_shock_FTO_m6AIP-rep1 vs Untreated_FTO_m6AIP-rep1""","""GSM5456687 vs GSM5456701""","""Public on Jul 19 2021""","""Jul 16 2021""","""Jul 20 2021""","""SRA""","""1""","""HEK293T""","""Homo sapiens""","""9606""","""polyA RNA""","""In vitro demethylation treatment""","""GPL20795""","""Kai,,Li""","""li_kai@pku.edu.cn""","""Yi Lab""","""School of Life Science""","""Peking University""","""Yi He Yuan Street No.5""","""BeiJing""","""100871""","""China""","""HiSeq X Ten""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5456687 vs GSM5456701""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5456nnn/GSM5456687/suppl/GSM5456687_HS-FTO_m6AIP-1_S1_L008_genome.bam_downsample.sort.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5456nnn/GSM5456701/suppl…",…,"""https://doi.org/10.3389/fonc.2021.612354""","""SRR15177493 vs SRR15177507""","""HiSeq X Ten sequencing: GSM5456687: heat_shock_FTO_m6AIP-rep1 Homo sapiens RNA-Seq vs HiSeq X Ten sequencing: GSM5456701: Untreated_FTO_m6AIP-rep1 Homo sapiens RNA-Seq""","""m6Am-seq reveals the dynamic m6Am methylation in the human transcriptome""","""SRA1260268""","""cDNA""","""2021-07-24""","""1516175044 vs 2035825226""","""1960529949 vs 2658177995""","""ILLUMINA""","""Generic""","""heat_shock_FTO_m6AIP-rep1 vs Untreated_FTO_m6AIP-rep1""","""c08bac270da863069c4c4cc8f104a81d vs 4d648fecfbdacf44edfa1fafd1097ab8""","""SRP328696""","""26450261 vs 33928166""","""m6Am-seq reveals the dynamic m6Am methylation in the human transcriptome""","""heat_shock_FTO_m6AIP-rep1 vs Untreated_FTO_m6AIP-rep1""","""HiSeq X Ten""","""HiSeq X Ten sequencing: GSM5456687: heat_shock_FTO_m6AIP-rep1 Homo sapiens RNA-Seq vs HiSeq X Ten sequencing: GSM5456701: Untreated_FTO_m6AIP-rep1 Homo sapiens RNA-Seq""","""699573d9761f8be94ae9388156c7a5bb vs b0bef922e1e4a3e468ded2eab84ddcc0""","""human embryonic kidney""","""3967539150 vs 5089224900""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM5456687 vs GSM5456701""","""GSM5456687_r1 vs GSM5456701_r1""","""Total RNA was extracted from tissues using TRIzol reagent (Invitrogen,15596018), followed by DNase I (NEB, M0303L) treatment to remove DNA contamination. Additional phenol-chloroform isolation and eth…","""GSM5456687 

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE70485 (SINGLE) m1A-seq

- validation date: 2025-05-06
- [The dynamic N1-methyladenosine methylome in eukaryotic messenger RNA](https://doi.org/10.1038/nature16998)
- m1A-seqを行っているデータ
- 熱ショックを加えているため、念の為収集する
- `HEPG2 mRNA heat shock 4h IP`, `HEPG2 mRNA heat shock 4h input`, `hepg2_mrna_untreated_ip_replicate_b`, `hepg2_mrna_untreated_input_replicate_b`の4つを念の為収集
- ※ 他にもreplicate C~E があるが今回は使用しない
- 処理を[./12_4_metadata_modified_false_no_control.ipynb](./12_4_metadata_modified_false_no_control.ipynb)から移動

In [54]:
GSE70485_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE70485/GSE70485_PEP/GSE70485_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("organism") == "Homo sapiens") &
    ~(
        (pl.col("sample_title").str.contains(r'HEPG2 mRNA Glucose starvation 4h')) | 
        (pl.col("sample_title").str.contains(r'HepG2 PARSseq')) | 
        (pl.col("sample_title").str.contains(r'replicate [C-E]')) | # Use regex character class [A-D] to match replicate C, D, E
        (pl.col("sample_title").str.contains(r'HEPG2 total RNA rearranged at [5-6]0c IP')) |
        (pl.col("sample_title").str.contains(r'HEPG2 total RNA rearranged at [5-6]0c input')) |
        (pl.col("sample_title").str.contains(r'HEK293')) |
        (pl.col("sample_title").str.contains(r'HeLa mRNA')) |
        (pl.col("sample_title") == "HEPG2 total RNA Untreated IP competitive elution") |
        (pl.col("sample_title") == "HEPG2 total RNA Untreated input competitive elution") |
        (pl.col("sample_title") == "HEPG2 total RNA Untreated IP replicate") |
        (pl.col("sample_title") == "HEPG2 total RNA Untreated input replicate")
    )
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("NEBNext Ultra RNA Library Prep Kit").alias("rna_library"), # add rna_library column from GEO page (New England Biolabs)
    pl.lit("https://doi.org/10.1038/nature16998").alias("reference_doi") # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE70485 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE70485_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE70485_iseq_filtered = GSE70485[[s.name for s in GSE70485 if not (s.null_count() == GSE70485.height)]]


# display(GSE70485_PEP)

In [55]:
GSE70485_combined = GSE70485_PEP.join(
    GSE70485_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE70485_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE70485_modified_metadata.tsv",
    separator='\t'
)

# display(GSE70485_combined)

In [56]:
defined_pairs_gse70485_m1a_seq = [
    ("SRR3000786", "SRR3000774"), # HEPG2 mRNA heat shock 4h IP vs HEPG2 mRNA untreated IP replicate B
    ("SRR3000787", "SRR3000775")  # HEPG2 mRNA heat shock 4h input vs HEPG2 mRNA untreated input replicate B
]

paired_metadata_df_gse70485 = create_paired_metadata(
    combined_df=GSE70485_combined,
    defined_sample_pairs=defined_pairs_gse70485_m1a_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse70485 is not None:
    paired_metadata_df_gse70485.write_csv(
        "../../Data/metadata/metadata_paired/GSE70485_m1a_seq_paired_metadata.tsv",
        separator='\t'
    )

# display(paired_metadata_df_gse70485)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE218180 (PAIRED) FANCI

- validation date: 2025-05-06
- [RNA damage compartmentalization by DHX9 stress granules](https://doi.org/10.1016/j.cell.2024.02.028)
- DHX9というヘリカーゼが関与する stress granuleに関する研究
- RNA-seqのデータの他に`fluorescence-activated non-membrane condensates isolation (FANCI)`というSGをフィルタリングする手法も含まれている
- FANCIのサンプルデータに熱ストレス曝露のデータが含まれており, `sample_library_source`を見ると transcriptomicになっており、悩みどころである
- **後でもう一度判断するとして、念の為収集しておく**
- 実験条件は43℃, 60min. (論文より)
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

In [57]:
GSE218180_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE218180/GSE218180_PEP/GSE218180_PEP_raw.csv",
    separator=','
).filter(
    ((pl.col("treatment") == "untreated") | # untreated
    (pl.col("treatment") == "Hs")) & # Heat shock
    (pl.col("sample_title").str.contains(r"FANCI")) # FANCI
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("Illumina's Stranded Total RNA with RiboZero Plus kit").alias("rna_library"), # add rna_library column from GEO page (Illumina)
    pl.lit("https://doi.org/10.1016/j.cell.2024.02.028").alias("reference_doi") # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE218180 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE218180_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE218180_iseq_filtered = GSE218180[[s.name for s in GSE218180 if not (s.null_count() == GSE218180.height)]]

# display(GSE218180_PEP)

In [58]:
GSE218180_combined = GSE218180_PEP.join(
    GSE218180_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE218180_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE218180_fanci_modified_metadata.tsv",
    separator='\t'
)

# display(GSE218180_combined)

In [59]:
defined_pairs_gse218180 = [
    ("SRR22319541", "SRR22319547"), # FANCI, Total RNA from Hela with heat shock(Hs) stress, 1 vs FANCI, Total RNA from Hela, 1
    ("SRR22319540", "SRR22319546"), # FANCI, Total RNA from Hela with heat shock(Hs) stress, 2 vs FANCI, Total RNA from Hela, 2
    ("SRR22319539", "SRR22319545"), # FANCI, Total RNA from Hela with heat shock(Hs) stress, 3 vs FANCI, Total RNA from Hela, 3
    ("SRR22319523", "SRR22319547"), # FANCI, Total RNA from SG with heat shock(Hs) stress, 1 vs FANCI, Total RNA from Hela, 1
    ("SRR22319522", "SRR22319546"), # FANCI, Total RNA from SG with heat shock(Hs) stress, 2 vs FANCI, Total RNA from Hela, 2
    ("SRR22319521", "SRR22319545")  # FANCI, Total RNA from SG with heat shock(Hs) stress, 3 vs FANCI, Total RNA from Hela, 3
]

paired_metadata_df_gse218180_fanci = create_paired_metadata(
    combined_df=GSE218180_combined,
    defined_sample_pairs=defined_pairs_gse218180,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse218180_fanci is not None:
    paired_metadata_df_gse218180_fanci.write_csv(
        "../../Data/metadata/metadata_paired/GSE218180_fanci_paired_metadata.tsv",
        separator="\t"
    )

display(paired_metadata_df_gse218180_fanci)

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,…,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR22319541""","""SRR22319547""","""fanci_total_rna_from_hela_with_heat_shock_hs_stress_1 vs fanci_total_rna_from_hela_1""","""null""","""null""","""null""","""null""","""null""","""null""","""FANCI, Total RNA from Hela with heat shock(Hs) stress, 1 vs FANCI, Total RNA from Hela, 1""","""GSM6736098 vs GSM6736092""","""Public on Nov 11 2023""","""Nov 17 2022""","""Nov 11 2023""","""SRA""","""1""","""Hela""","""Homo sapiens""","""9606""","""total RNA""","""FANCI (During processing it is named as s7), featureCounts_exon_counts.tsv, featureCounts_geneBody_counts.tsv, batch_totalRNA_mock_HS_counts_DESeq2.normalized.tsv vs FANCI (During processing it is nam…","""GPL24676""","""Asifa,,Akhtar""","""akhtarlab_data@ie-freiburg.mpg.de""","""Akhtar Lab""","""Chromatin Regulation""","""Stuebeweg 51""","""Freiburg""","""79108""","""Germany""","""Illumina NovaSeq 6000""","""other""","""transcriptomic""","""OTHER""","""GSM6736098 vs GSM6736092""","""NONE""","""GSE218180""",…,"""SRR22319541 vs SRR22319547""","""Illumina NovaSeq 6000 sequencing: GSM6736098: FANCI, Total RNA from Hela with heat shock(Hs) stress, 1 Homo sapiens OTHER vs Illumina NovaSeq 6000 sequencing: GSM6736092: FANCI, Total RNA from Hela, 1…","""RNA damage compartmentalization by DHX9 granules""","""SRA1542297""","""other""","""2023-11-26""","""1743740155 vs 1467950664""","""815757930;908893487 vs 672502364;755416798""","""ILLUMINA""","""missing""","""Generic""","""FANCI, Total RNA from Hela with heat shock(Hs) stress, 1 vs FANCI, Total RNA from Hela, 1""","""5cc37accdbabfd863648462a055eda88;edabcabe0a2129ff7c0da6444caa34c2 vs e9363d76356a21af362b767ba4392f10;2ef82fb610f2c463541f15ed6b651cdc""","""SRP408398""","""38361568 vs 28975555""","""RNA damage compartmentalization by DHX9 granules""","""FANCI, Total RNA from Hela with heat shock(Hs) stress, 1 vs FANCI, Total RNA from Hela, 1""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM6736098: FANCI, Total RNA from Hela with heat shock(Hs) stress, 1 Homo sapiens OTHER vs Illumina NovaSeq 6000 sequencing: GSM6736092: FANCI, Total RNA from Hela, 1…","""1cb5581955353111f0ff6c8635190bc3 vs 1c0753600c89676fb3a9caeb4688a445""","""7749036736 vs 5853062110""","""OTHER""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM6736098 vs GSM6736092""","""GSM6736098_r1 vs GSM6736092_r1""","""Total RNA from cells was collected using Trizol and purified using Zymo Research DirectZol RNA Mini Prep Kit. Total RNA f

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# PRJDB9865 CAGE-seq (SINGLE)

&nbsp;

- validate date 2025-05-18
- [Heat shock induces the nuclear accumulation of YAP1 via SRC](https://doi.org/10.1016/j.yexcr.2020.112439)
- CAGE-seqのデータ
- なにかに使用できる可能性が出てきたため、収集する
- ("DRR228710", "DRR228707")  # U2OS siYAP1 / HeatShock 2h vs U2OS parent cell / HeatShock 0h のペア構築は少し怪しいが少ないので集めておこう
- [12_5_metadata_modified_bioproject_biostudies.ipynb](./12_5_metadata_modified_bioproject_biostudies.ipynb)から移動
- (2026-07-23) [ChIP-Atlas](https://chip-atlas.org/) にはデータが登録されていない

In [60]:
PRJDB9865 = pl.read_csv(
    "../../Data/metadata/metadata_bioproject/PRJDB9865_metadata.tsv",
    separator="\t"
).with_columns(
    pl.lit("nAnT-iCAGE protocol").alias("rna_library"), # add rna_library column from article,
    pl.lit("https://doi.org/10.1016/j.yexcr.2020.112439").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("120 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
).sort(
    "run_accession",
    descending=False
).drop(
    drop_columns_list
)

PRJDB9865_filtered = PRJDB9865[[s.name for s in PRJDB9865 if not (s.null_count() == PRJDB9865.height)]]

PRJDB9865_filtered.write_csv(
    "../../Data/metadata/metadata_bioproject/PRJDB9865_modified_metadata.tsv",
    separator="\t"
)

display(PRJDB9865_filtered)

run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,sample_title,instrument_model,description,sra_md5,cell_type,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,experiment_alias,center_name,cell_line,first_created,experiment_accession,study_accession,accession,secondary_sample_accession,first_public,study_alias,rna_library,reference_doi,stress treatment temperature,stress treatment time
str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR228707""","""NextSeq 500 sequencing: NextSeq 500 sequencing of SAMD00226153""","""SAMD00226153""","""Homo sapiens""","""DRA010231""","""CAGE""","""2020-05-23""",534626471,724165620,"""ILLUMINA""","""Generic""","""U2OS parent cell / HeatShock 0h""","""83dfceb008ae6265b0784f51d023e867""","""DRP006137""",17527966,"""Gene expression data of YAP1-knockdowned U2OS cells under heat shock stress""","""U2OS parent cell / HeatShock 0h""","""NextSeq 500""","""NextSeq 500 sequencing: NextSeq 500 sequencing of SAMD00226153""","""622f90a51fb117077679de80dc22750b""","""bone osteosarcoma""",1323139093,"""Targeted-Capture""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00226153""","""DRR228707""","""DRX218962""","""TOKYO_MEDEN""","""U2OS""","""2020-05-23""","""DRX218962""","""PRJDB9865""","""DRR228707""","""DRS139458""","""2020-05-23""","""DRP006137""","""nAnT-iCAGE protocol""","""https://doi.org/10.1016/j.yexcr.2020.112439""","""42 C""","""120 min."""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""DRR228710""","""NextSeq 500 sequencing: NextSeq 500 sequencing of SAMD00226156""","""SAMD00226156""","""Homo sapiens""","""DRA010231""","""CAGE""","""2020-05-23""",524144907,715508119,"""ILLUMINA""","""Generic""","""U2OS siYAP1 / HeatShock 2h""","""9aa320602d3bd07ab4946fa0de926aed""","""DRP006137""",17262516,"""Gene expression data of YAP1-knockdowned U2OS cells under heat shock stress""","""U2OS siYAP1 / HeatShock 2h""","""NextSeq 500""","""NextSeq 500 sequencing: NextSeq 500 sequencing of SAMD00226156""","""c2b012e0745e881ccddb52049dd9f43e""","""bone osteosarcoma""",1302043188,"""Targeted-Capture""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00226156""","""DRR228710""","""DRX218965""","""TOKYO_MEDEN""","""U2OS""","""2020-05-23""","""DRX218965""","""PRJDB9865""","""DRR228710""","""DRS139461""","""2020-05-23""","""DRP006137""","""nAnT-iCAGE protocol""","""https://doi.org/10.1016/j.yexcr.2020.112439""","""42 C""","""120 min."""


In [61]:
defined_pairs_prjdb9865 = [
    ("DRR228709", "DRR228708"), # U2OS siCtrl / HeatShock 2h vs U2OS siCtrl / HeatShock 0h
    ("DRR228710", "DRR228707")  # U2OS siYAP1 / HeatShock 2h vs U2OS parent cell / HeatShock 0h
]

paired_metadata_df_prjdb9865 = create_paired_metadata(
    combined_df=PRJDB9865_filtered,
    defined_sample_pairs=defined_pairs_prjdb9865,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_prjdb9865 is not None:
    paired_metadata_df_prjdb9865.write_csv(
        "../../Data/metadata/metadata_paired/PRJDB9865_cage_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_prjdb9865)
else:
    print("No paired metadata created for PRJDB9865")

sample_heat_id,sample_control_id,run_accession,experiment_title,sample_accession,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,sample_title,instrument_model,description,sra_md5,cell_type,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,experiment_alias,center_name,cell_line,first_created,experiment_accession,study_accession,secondary_sample_accession,first_public,study_alias,rna_library,reference_doi,stress treatment temperature,stress treatment time
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""DRR228709""","""DRR228708""","""DRR228709 vs DRR228708""","""NextSeq 500 sequencing: NextSeq 500 sequencing of SAMD00226155 vs NextSeq 500 sequencing: NextSeq 500 sequencing of SAMD00226154""","""SAMD00226155 vs SAMD00226154""","""Homo sapiens""","""DRA010231""","""CAGE""","""2020-05-23""","""526718509 vs 617915475""","""714110332 vs 836137403""","""ILLUMINA""","""Generic""","""U2OS siCtrl / HeatShock 2h vs U2OS siCtrl / HeatShock 0h""","""20f59783e9e03729517de27dd99a9196 vs fa0da776b4a8a3be7a7ee405e2f36f37""","""DRP006137""","""17339074 vs 20094977""","""Gene expression data of YAP1-knockdowned U2OS cells under heat shock stress""","""U2OS siCtrl / HeatShock 2h vs U2OS siCtrl / HeatShock 0h""","""NextSeq 500""","""NextSeq 500 sequencing: NextSeq 500 sequencing of SAMD00226155 vs NextSeq 500 sequencing: NextSeq 500 sequencing of SAMD00226154""","""c93031e114981d05e0dae9621497f835 vs 4da6ef98d59f3e02f1a946579b35b0a5""","""bone osteosarcoma""","""1307428627 vs 1516836738""","""Targeted-Capture""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00226155 vs SAMD00226154""","""DRR228709 vs DRR228708""","""DRX218964 vs DRX218963""","""TOKYO_MEDEN""","""U2OS""","""2020-05-23""","""DRX218964 vs DRX218963""","""PRJDB9865""","""DRS139460 vs DRS139459""","""2020-05-23""","""DRP006137""","""nAnT-iCAGE protocol""","""https://doi.org/10.1016/j.yexcr.2020.112439""","""42 C""","""120 min."""
"""DRR228710""","""DRR228707""","""DRR228710 vs DRR228707""","""NextSeq 500 sequencing: NextSeq 500 sequencing of SAMD00226156 vs NextSeq 500 sequencing: NextSeq 500 sequencing of SAMD00226153""","""SAMD00226156 vs SAMD00226153""","""Homo sapiens""","""DRA010231""","""CAGE""","""2020-05-23""","""524144907 vs 534626471""","""715508119 vs 724165620""","""ILLUMINA""","""Generic""","""U2OS siYAP1 / HeatShock 2h vs U2OS parent cell / HeatShock 0h""","""9aa320602d3bd07ab4946fa0de926aed vs 83dfceb008ae6265b0784f51d023e867""","""DRP006137""","""17262516 vs 17527966""","""Gene expression data of YAP1-knockdowned U2OS cells under heat shock stress""","""U2OS siYAP1 / HeatShock 2h vs U2OS parent cell / HeatShock 0h""","""NextSeq 500""","""NextSeq 500 sequencing: NextSeq 500 sequencing of SAMD00226156 vs NextSeq 500 sequencing: NextSeq 500 sequencing of SAMD00226153""","""c2b012e0745e881ccddb52049dd9f43e vs 622f90a51fb117077679de80dc22750b""","""bone osteosarcoma""","""1302043188 vs 1323139093""","""Targeted-Capture""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""SAMD00226156 vs SAMD00226153""","""DRR228710 vs DRR228707""","""DRX218965 vs DRX218962""","""TOKYO_MEDEN""","""U2OS""","""2020-05-23""","""DRX218965 vs DRX218962""","""PRJDB9865""","""DRS139461 vs DRS139458""","""2020-05-23""","""DRP006137""","""nAnT-iCAGE protocol""","""https://doi.org/10.1016/j.yexcr.2020.112439""","""42 C""","""120 min."""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## PRJEB73803 (ERP158539) (PAIRED) 4sU-seq

- validate date: 2025-05-18
- [E-MTAB-13903](https://www.ebi.ac.uk/biostudies/ArrayExpress/studies/E-MTAB-13903?query=E-MTAB-13903)
- 論文は見つけられなかったが、熱ショックとコントロールのデータが登録されているので念の為収集する
- `control exposed to 200nM 4sU` と `HS-treated exposed to 200nM 4sU` の2つがある
- [./12_5_metadata_modified_bioproject_biostudies.ipynb](./12_5_metadata_modified_bioproject_biostudies.ipynb)から処理を移動

In [62]:
PRJEB73803 = pl.read_csv(
    "../../Data/metadata/metadata_biostudies/ERP158539.metadata.tsv",
    separator="\t"
).with_columns(
    pl.lit("unknown").alias("rna_library"), # add rna_library column from ENA webpage
    pl.lit("unknown").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).drop(
    drop_columns_list
)

PRJEB73803_filtered = PRJEB73803[[s.name for s in PRJEB73803 if not (s.null_count() == PRJEB73803.height)]]

PRJEB73803_filtered.write_csv(
    "../../Data/metadata/metadata_bioproject/PRJEB73803_modified_metadata.tsv",
    separator="\t"
)


# display(PRJEB73803_filtered)

In [63]:
defined_pairs_prjeb73803 = [
    ("ERR12737081", "ERR12737079"), # E-MTAB-13903:MCF10A_HS_2h vs E-MTAB-13903:MCF10A_CTR_2h
    ("ERR12737082", "ERR12737080"), # E-MTAB-13903:MCF10A_HS_4h vs E-MTAB-13903:MCF10A_CTR_4h
    ("ERR12737085", "ERR12737083"), # E-MTAB-13903:MCF7_HS_2h vs E-MTAB-13903:MCF7_CTR_2h
    ("ERR12737086", "ERR12737084")  # E-MTAB-13903:MCF7_HS_4h vs E-MTAB-13903:MCF7_CTR_4h
]

paired_metadata_df_prjeb73803 = create_paired_metadata(
    combined_df=PRJEB73803_filtered,
    defined_sample_pairs=defined_pairs_prjeb73803,
    sample_id_column_name="accession"
)

if paired_metadata_df_prjeb73803 is not None:
    paired_metadata_df_prjeb73803.write_csv(
        "../../Data/metadata/metadata_paired/prjeb73803_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_prjeb73803)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,run_accession,experiment_title,sample_accession,project_name,submission_accession,bam_md5,library_selection,last_updated,submitted_galaxy,submitted_ftp,fastq_bytes,instrument_platform,submitted_aspera,bam_ftp,submitted_md5,checklist,submitted_bytes,sample_description,fastq_md5,secondary_study_accession,read_count,study_title,sample_title,instrument_model,bam_bytes,description,cell_type,broker_name,base_count,library_strategy,library_source,library_layout,dev_stage,bam_aspera,datahub,status,tax_lineage,sample_alias,run_alias,submitted_read_type,library_construction_protocol,collection_date,experiment_alias,center_name,cell_line,library_name,first_created,experiment_accession,scientific_name,tax_id,study_accession,submitted_format,bam_galaxy,secondary_sample_accession,first_public,study_alias,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""ERR12737081""","""ERR12737079""","""ERR12737081 vs ERR12737079""","""Illumina NovaSeq 6000 sequencing: Searching for the fusion transcripts created during proteotoxic stress""","""SAMEA115398848 vs SAMEA115398846""","""Searching for the fusion transcripts created during proteotoxic stress""","""ERA29475593""","""f8378242b3ea97bd40094c34c6bbf42d vs 781c08ec96107a27343b66acbe61ec65""","""Inverse rRNA""","""2024-12-01""","""ftp.sra.ebi.ac.uk/vol1/run/ERR127/ERR12737081/MCF10_HS_2h_ZKRN240001585-1A_HVV7YDSX7_L4_1.fq.gz;ftp.sra.ebi.ac.uk/vol1/run/ERR127/ERR12737081/MCF10_HS_2h_ZKRN240001585-1A_HVV7YDSX7_L4_2.fq.gz vs ftp.s…","""ftp.sra.ebi.ac.uk/vol1/run/ERR127/ERR12737081/MCF10_HS_2h_ZKRN240001585-1A_HVV7YDSX7_L4_1.fq.gz;ftp.sra.ebi.ac.uk/vol1/run/ERR127/ERR12737081/MCF10_HS_2h_ZKRN240001585-1A_HVV7YDSX7_L4_2.fq.gz vs ftp.s…","""8137749953;8257474400 vs 7215199345;7176549660""","""ILLUMINA""","""fasp.sra.ebi.ac.uk:/vol1/run/ERR127/ERR12737081/MCF10_HS_2h_ZKRN240001585-1A_HVV7YDSX7_L4_1.fq.gz;fasp.sra.ebi.ac.uk:/vol1/run/ERR127/ERR12737081/MCF10_HS_2h_ZKRN240001585-1A_HVV7YDSX7_L4_2.fq.gz vs f…","""ftp.sra.ebi.ac.uk/vol1/err/ERR127/081/ERR12737081/ERR12737081.bam vs ftp.sra.ebi.ac.uk/vol1/err/ERR127/079/ERR12737079/ERR12737079.bam""","""8113bb0b3cfe99a30dd807b004c205f7;66cc689cbf331fa6cd9cd167dcadd078 vs 6639ef3b383f359248114db22407661b;6da6d6d91bf6b492c1d617869239e0da""","""ERC000011""","""8847196938;9028518552 vs 7837775486;7835141478""","""Protocols: Non-tumorigenic human breast epithelial MCF10A cells (authenticated in 2016) were cultured in DMEM/F12 medium supplemented with 5% horse serum (BioWest, Nuaillé, France), 5 µg/mL insulin (S…","""019bb33521772cc8edb35b129fb205cf;24376c13ad344d5540f1ca4353eedd15 vs bace738d3ab600216d1e3161a0cbbea5;51a329835ee610e65bd85e90ab646c11""","""ERP158539""","""243664890 vs 214223366""","""Searching for the fusion transcripts created during proteotoxic stress""","""MCF10A_HS_2h vs MCF10A_CTR_2h""","""Illumina NovaSeq 6000""","""15933898254 vs 13961870472""","""Illumina NovaSeq 6000 sequencing: Searching for the fusion transcripts created during proteotoxic stress""","""epithelial cell""","""ArrayExpress""","""36549733500 vs 32133504900""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""adult""","""fasp.sra.ebi.ac.uk:/vol1/err/ERR127/081/ERR12737081/ERR12737081.bam vs fasp.sra.ebi.ac.uk:/vol1/err/ERR127/079/ERR12737079/ERR12737079.bam""","""dcc_arrayexpress""","""public""","""1;131567;2759;33154;33208;6072;33213;33511;7711;89593;7742;7776;117570;117571;8287;1338369;32523;32524;40674;32525;9347;1437010;314146;9443;376913;314293;9526;314295;9604;207598;9605;9606""","""E-MTAB-13903:MCF10A_HS_2h vs E-MTAB-13903:MCF10A_CTR_2h""","""E-MTAB-13903:MCF10_HS_2h_ZKRN240001585-1A_HVV7YDSX7_L4_ vs E-MTAB-13903:MCF10_CTR_2h_ZKRN2400

&nbsp;

&nbsp;

&nbsp;

---

&nbsp;

&nbsp;

&nbsp;

### [../../Data/curation/GEO_other_result/gds_other_result_summary_260123_modified.md](../../Data/curation/GEO_other_result/gds_other_result_summary_260123_modified.md)から移行したヒット

- [../../Data/curation/GEO_other_result/gds_other_result_summary_260123_modified.md](../../Data/curation/GEO_other_result/gds_other_result_summary_260123_modified.md)で、PRO-seqなどではないものの、今後使用できそうなデータを以下に記載する

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE63655 (SINGLE) Cap-seq

- validate date: 2026-02-21
- 熱ストレスに関連するPRO-seqを探索した結果で発見されたデータ
- - [gds_other_result_summary_260123_modified_proseq.md](../../Data/curation/GEO_other_result/gds_other_result_summary_260123_modified_proseq.md)
- [Chemical pulldown reveals dynamic pseudouridylation of the mammalian transcriptome](https://doi.org/10.1038/nchembio.1836)
- `Cap-seq`という最近見ない手法
- [illumina: CAP-seq](https://jp.illumina.com/science/sequencing-method-explorer/kits-and-arrays/cap-seq.html)
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動

In [64]:
GSE63655_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE63655/GSE63655_PEP/GSE63655_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_organism_ch1") == "Homo sapiens")
)

display(GSE63655_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,cell_type,enrichment,barcode,library_strategy,genome_build,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str
"""wt_input_rep1""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR1663493""","""SRX768619""","""WT_Input_Rep1""","""GSM1554812""","""Public on Jun 05 2015""","""Nov 26 2014""","""May 15 2019""","""SRA""",1,"""WT_Input""","""Homo sapiens""",9606,"""polyA RNA""","""GPL16791""","""Ping,,Zhu""","""pingzhu.work@gmail.com""","""State Key Laboratory of Experimental Hematology""","""288, Nanjing Road, Heping District""","""Tianjin""",300020,"""China""","""Illumina HiSeq 2500""","""other""","""transcriptomic""","""OTHER""","""GSM1554812""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1554nnn/GSM1554812/suppl/GSM1554812_WT_Input_Rep1.txt.gz""","""GSE63655""",0,"""SRX768619""","""HEK293T""","""human embryonic kindey cells""","""none""","""NNNNCGATNN""","""CAP-seq""","""hg19, mm9""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN03222655""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX768619"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ko_pulldown_rep2""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR1663518""","""SRX768644""","""KO_Pulldown_Rep2""","""GSM1554837""","""Public on Jun 05 2015""","""Nov 26 2014""","""May 15 2019""","""SRA""",1,"""KO_Pulldown""","""Homo sapiens""",9606,"""polyA RNA""","""GPL16791""","""Ping,,Zhu""","""pingzhu.work@gmail.com""","""State Key Laboratory of Experimental Hematology""","""288, Nanjing Road, Heping District""","""Tianjin""",300020,"""China""","""Illumina HiSeq 2500""","""other""","""transcriptomic""","""OTHER""","""GSM1554837""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM1554nnn/GSM1554837/suppl/GSM1554837_KO_Pulldown_Rep2.txt.gz""","""GSE63655""",0,"""SRX768644""","""HEK293T""","""human embryonic kindey cells""","""N3-CMC treat and pull down""","""NNNNTAGCNN""","""CAP-seq""","""hg19, mm9""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN03222670""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX768644"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE73405 (SINGLE) par-iclip / m6A profiling

- [5′ UTR m6A Promotes Cap-Independent Translation](https://doi.org/10.1016/j.cell.2015.10.012)
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動

In [65]:
GSE73405_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE73405/GSE73405_PEP/GSE73405_PEP_raw.csv",
    separator=','
)

display(GSE73405_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,cell_type,antibody_name,genome_build,supplementary_files_format_and_content,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str
"""eif3a_par-iclip_replicate1""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR2517447""","""SRX1287379""","""eif3a_par-iclip_replicate1""","""GSM1892896""","""Public on Oct 26 2015""","""Sep 24 2015""","""May 15 2019""","""SRA""",1,"""HEK293T cells""","""Homo sapiens""",9606,"""10 million cells were incubated with 100 mM 4-thiouridine for 8 h and irradiated with 354 nm UV light prior to harvesting.""","""total RNA""","""Cell lysates were isolated and crosslinked eIF3:RNA complexes were immunoprecipitated with anti-eIF3a antibody., Library construction was performed as per the method of Huppertz et al., iCLIP: protein…","""Barcodes used were NNNGGTTNN (rep1) and NNNTTGTNN (rep2), Alignment to hg19 was performed using bowtie""","""GPL11154""","""Kate,,Meyer""","""kate.meyer@duke.edu""",9196849562,"""Biochemistry""","""Duke University School of Medicine""","""307 Research Drive Box 3711""","""D""","""NC""",27710,"""USA""","""Illumina HiSeq 2000""","""other""","""transcriptomic""","""RIP-Seq""","""GSM1892896""","""NONE""","""GSE73405""",0,"""SRX1287379""","""HEK293T""","""human embryonic kidney cells""","""eIF3a (Abcam #ab86146)""","""hg19""","""Peak-calling for processed files was performed using Pyicoclip""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN04110551""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX1287379"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""hepg2_control_2""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR3066594""","""SRX1508526""","""HepG2 Control 2""","""GSM2011456""","""Public on Dec 31 2015""","""Dec 30 2015""","""May 15 2019""","""SRA""",1,"""HepG2 cells""","""Homo sapiens""",9606,"""Cells were grown in standard growth media for each cell line""","""polyA RNA""","""Total RNA was extracted using TRIzol reagent. RNA was then subjected to DNAseI treatment to remove any contaminating DNA. mRNA was then isolated using oligo(dT) Dynabeads (Invitrogen)., Library constr…","""Barcodes used were NNNGGTTNN (HSreplicates1), NNNTTGTNN (HSrep2), NNNCAATNN (CTLrep1), and NNNACCTNN (CTLrep2), Alignment to hg19 was performed using novoalign""","""GPL16791""","""Kate,,Meyer""","""kate.meyer@duke.edu""",9196849562,"""Biochemistry""","""Duke University School of Medicine""","""307 Research Drive Box 3711""","""D""","""NC""",27710,"""USA""","""Illumina HiSeq 2500""","""other""","""transcriptomic""","""OTHER""","""GSM2011456""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2011nnn/GSM2011456/suppl/GSM2011456_HepG2CTL2_processed.bedgraph.gz""","""GSE73405""",0,"""SRX1508526""","""HepG2""","""human liver hepatocellular carcinoma cells""","""Rabbit anti-m6A (Abcam #ab151230)""","""hg19""","""m6A sites were identified from single reads using CIMS (C2T transitions, p=0.1). Metagene profiles were performed using hg19 genomic annotations. For HEK control samples, miCLIP data from Linder et al…","""https:/

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE75418 CLIP-seq

- [iCLIP identifies novel roles for SAFB1 in regulating RNA processing and neuronal function](https://doi.org/10.1186/s12915-015-0220-7)
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動
- 熱ストレスに関連していない可能性があるな... と思っていたが、この研究で注目しているSAFB1 には以下のような記述があった
- `Following heat stress, SAFB1 translocates to nuclear stress bodies together with HSF1 and Sam68. It has been hypothesised that nuclear stress bodies may sequester splicing factors to downregulate normal cellular splicing and facilitate the processing of RNA transcripts that are essential for the stress response [10, 24].`
- つまり、このSAFB1の結合部位などを調べることができれば、よりlncRNA-miRNA-mRNAの解析に有用である可能性があるため、収集する

In [66]:
GSE75418_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE75418/GSE75418_PEP/GSE75418_PEP_raw.csv",
    separator=','
)

display(GSE75418_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_type,antibody,library_strategy,genome_build,supplementary_files_format_and_content,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str
"""1_safb1_iclip""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR2962747""","""SRX1453676""","""1 SAFB1 iCLIP""","""GSM1954829""","""Public on Dec 01 2015""","""Nov 25 2015""","""May 15 2019""","""SRA""",1,"""SHSY-5Y""","""Homo sapiens""",9606,"""SHSY-5Y cells were grown using standard cell culture techniques in DMEM-F12 media supplemented with 10% FBS""","""total RNA""","""1 SAFB1""","""GPL16791""","""James,,Uney""","""james.uney@bristol.ac.uk""",1173311466,"""School of Clinical Sciences""","""University of Bristol""","""Medical School, University Walk""","""Bristol""","""BS8 1TD""","""United Kingdom""","""Illumina HiSeq 2500""","""other""","""transcriptomic""","""OTHER""","""GSM1954829""","""NONE""","""GSE75418, GSE75469""",0,"""SRX1453676""","""Neuroblastoma""","""Bethyl SAFB1""","""CLIP-Seq""","""hg19/GRCh37""","""Gene coordinates (columns 1-4), gene name, gene segment type and biotype, and tag sums for each expt""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN04299703""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX1453676"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""6_safb1_iclip""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR2984713""","""SRX1473616""","""6 SAFB1 iCLIP""","""GSM1969233""","""Public on Dec 10 2015""","""Dec 10 2015""","""May 15 2019""","""SRA""",1,"""SHSY-5Y""","""Homo sapiens""",9606,"""SHSY-5Y cells were grown using standard cell culture techniques in DMEM-F12 media supplemented with 10% FBS (4 SAFB1, 5 SAFB1 & 6SAFB1 were exposed to a heat stress 42 degrees celcius)""","""total RNA""","""Stress 42oC""","""GPL16791""","""James,,Uney""","""james.uney@bristol.ac.uk""",1173311466,"""School of Clinical Sciences""","""University of Bristol""","""Medical School, University Walk""","""Bristol""","""BS8 1TD""","""United Kingdom""","""Illumina HiSeq 2500""","""other""","""transcriptomic""","""OTHER""","""GSM1969233""","""NONE""","""GSE75418, GSE75469""",0,"""SRX1473616""","""Neuroblastoma""","""Bethyl SAFB1""","""CLIP-Seq""","""hg19/GRCh37""","""Kmer Z-scores: z-score 1 processed from sample 1 SAFB1 (GSM1954829); z-score 2 processed from sample 2 SAFB1(GSM1954830) etc.""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN04329926""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX1473616"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE105028 (eU-seq, ~~Hi-C,~~ ~~Hi-ChIP,~~ ~~ChIP-seq~~, ~~ATAC-seq~~ )

- [Architectural Proteins and Pluripotency Factors Cooperate to Orchestrate the Transcriptional Response of hESCs to Temperature Stress](https://doi.org/10.1016/j.molcel.2018.07.012)
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動
- ATAC-seq のデータは、[./12_8_metadata_atacseq.ipynb](./12_8_metadata_atacseq.ipynb)に移動
- ChIP-seq のデータは、[./12_8_metadata_chipseq.ipynb](./12_8_metadata_chipseq.ipynb)に移動

In [67]:
GSE105028_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE105028/GSE105028_PEP/GSE105028_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("library_strategy") == "eU-Seq")
)

pl.Config.set_tbl_rows(38)
display(GSE105028_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,condition,library_strategy,genome_build,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str
"""eu-seq""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6177930""","""SRX3288552""","""eU-Seq""","""GSM2816611""","""Public on Aug 03 2018""","""Oct 16 2017""","""May 15 2019""","""SRA""",1,"""H9 hESCs""","""Homo sapiens""",9606,"""total RNA""","""eU-Seq under normal conditions. nascent transcribed RNA""","""GPL11154""","""XIAOWEN,,LYU""","""xiaowenlyu@gmail.com""","""+86-18701064502""","""XIAMEN UNIVERSITY""","""No.4221, Xiang'an South Road, Xiang'an District""","""XIAMEN""","""FUJIAN""",361102,"""China""","""Illumina HiSeq 2000""","""other""","""transcriptomic""","""OTHER""","""GSM2816611""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2816nnn/GSM2816611/suppl/GSM2816611_eU-Seq.bw""","""GSE105028""",0,"""SRX3288552""","""H9 human Embryonic Stem Cell Line""","""under normal conditions. nascent transcribed RNA""","""eU-Seq""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN07790041""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX3288552"""
"""hs_eu-seq""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6177931""","""SRX3288553""","""HS eU-Seq""","""GSM2816612""","""Public on Aug 03 2018""","""Oct 16 2017""","""May 15 2019""","""SRA""",1,"""H9 hESCs""","""Homo sapiens""",9606,"""total RNA""","""eU-Seq under heat shock conditions. nascent transcribed RNA""","""GPL11154""","""XIAOWEN,,LYU""","""xiaowenlyu@gmail.com""","""+86-18701064502""","""XIAMEN UNIVERSITY""","""No.4221, Xiang'an South Road, Xiang'an District""","""XIAMEN""","""FUJIAN""",361102,"""China""","""Illumina HiSeq 2000""","""other""","""transcriptomic""","""OTHER""","""GSM2816612""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2816nnn/GSM2816612/suppl/GSM2816612_HS_eU-Seq.bw""","""GSE105028""",0,"""SRX3288553""","""H9 human Embryonic Stem Cell Line""","""under heat shock conditions. nascent transcribed RNA""","""eU-Seq""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN07790040""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX3288553"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE105028 (Hi-C, Hi-ChIP, ~~eU-seq,~~ ~~ChIP-seq~~, ~~ATAC-seq~~ )

- [Architectural Proteins and Pluripotency Factors Cooperate to Orchestrate the Transcriptional Response of hESCs to Temperature Stress](https://doi.org/10.1016/j.molcel.2018.07.012)
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動
- ATAC-seq のデータは、[./12_8_metadata_atacseq.ipynb](./12_8_metadata_atacseq.ipynb)に移動
- ChIP-seq のデータは、[./12_8_metadata_chipseq.ipynb](./12_8_metadata_chipseq.ipynb)に移動
- [./12_8_metadata_hic.ipynb](./12_8_metadata_hic.ipynb)に移動

In [68]:
# GSE105028_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE105028/GSE105028_PEP/GSE105028_PEP_raw.csv",
#     separator=','
# ).filter(
#     (pl.col("library_strategy") != "ATAC-Seq") &
#     (pl.col("library_strategy") != "ChIP-Seq") &
#     (pl.col("library_strategy") != "eU-Seq")
# ).sort(
#     "srr",
#     descending=False
# )

# GSE105028_PEP.write_csv(
#     "../../Data/metadata/metadata_hic/GSE105028_PEP_raw_filtered.tsv",
#     separator='\t'
# )

# pl.Config.set_tbl_rows(38)
# display(GSE105028_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE130758 Hi-C

- validate date: 2026-02-22
- [Chromatin conformation remains stable upon extensive transcriptional changes driven by heat shock](https://doi.org/10.1073/pnas.1901244116)
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動
- [./12_8_metadata_hic.ipynb](./12_8_metadata_hic.ipynb)に移動

In [69]:
# GSE130758_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE130758/GSE130758_PEP/GSE130758_PEP_raw.csv",
#     separator=','
# )

# GSE130758_PEP.write_csv(
#     "../../out/Hi-C/GSE130758_PEP_raw_filtered.tsv",
#     separator='\t'
# )

# display(GSE130758_PEP)

&nbsp;

&nbsp;

&nbsp;

### GSE164200 (SINGLE) m6A-RIP-seq

- validate date: 2026-02-22
- [m6 A modification of HSATIII lncRNAs regulates temperature-dependent splicing](https://doi.org/10.15252/embj.2021107976)
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動
- [./12_7_metadata_meripseq.ipynb](./12_7_metadata_meripseq.ipynb)に処理を移動

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE169435 (SINGLE) Genome Sequence with MiNION

- validate date: 2026-02-22
- [Chimeric chromosome landscapes of human somatic cell cultures show dependence on stress and regulation of genomic repeats by CGGBP1](https://doi.org/10.18632/oncotarget.28174)
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動
- Library source が`genomic`で使えるかわからないが...

In [70]:
GSE169435_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE169435/GSE169435_PEP/GSE169435_PEP_raw.csv",
    separator=','
)

display(GSE169435_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_type,library_strategy,genome_build,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str
"""gm02639_-_37°c""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR14044873""","""SRX10421006""","""GM02639 - 37°C""","""GSM5203044""","""Public on Jan 10 2022""","""Mar 23 2021""","""Jan 10 2022""","""SRA""",1,"""Primary fibroblast""","""Homo sapiens""",9606,"""HEK293T cells were transduced with following shRNA lentiviruses: non-targeting (CT), CGGBP1-silencing (KD) [Origene Lentiviral system TL313955V]. For stable transduction, control and CGGBP1 shmiR tran…","""Human primary fibroblasts [Coriell Cell Repository GM02639] and HEK293T CT and KD [lentivirally transduced shRNA system as described in PMID 31547883] were cultured in DMEM supplemented with 10% fetal…","""genomic DNA""","""Human primary fibroblasts [Coriell Cell Repository GM02639] were harvested from two T25 flasks for each sample. HEK293T CT and KD cells were harvested from two 10cm dishes for each sample. Genomic DNA…","""The library were sequencd using MinKNOW v20.10.3 with real-time basecalling using Guppy v4.2.2., Sequencing adapters were trimmed using Porechop v0.2.4 from quality filtered reads and the reads were m…","""GPL24106""","""Umashankar,,Singh""","""HoMeCell Lab""","""Biological Sciences and Engineering""","""IIT Gandhinagar""","""Acad Block 4 Room 317 IIT Gandhinagar Palaj""","""Gandhinagar""","""Gujarat""",382055,"""India""","""MinION""","""other""","""genomic""","""OTHER""","""GSM5203044""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5203nnn/GSM5203044/suppl/GSM5203044_PrimaryFibroblasts_37C_fusion_A-B_coord.bed.gz""","""GSE169435""",0,"""SRX10421006""","""Human primary fibroblasts [Coriell Cell Repository GM02639]""","""Oxford Nanopore Long-read sequencing""","""hg38""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN18439514""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX10421006"""
"""gm02639_-_40°c""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR14044874""","""SRX10421007""","""GM02639 - 40°C""","""GSM5203045""","""Public on Jan 10 2022""","""Mar 23 2021""","""Jan 10 2022""","""SRA""",1,"""Primary fibroblast""","""Homo sapiens""",9606,"""HEK293T cells were transduced with following shRNA lentiviruses: non-targeting (CT), CGGBP1-silencing (KD) [Origene Lentiviral system TL313955V]. For stable transduction, control and CGGBP1 shmiR tran…","""Human primary fibroblasts [Coriell Cell Repository GM02639] and HEK293T CT and KD [lentivirally transduced shRNA system as described in PMID 31547883] were cultured in DMEM supplemented with 10% fetal…","""genomic DNA""","""Human primary fibroblasts [Coriell Cell Repository GM02639] were harvested from two T25 flasks for each sample. HEK293T CT and KD cells were harvested from two 10cm dishes for each sample. Genomic DNA…","""The library were sequencd using MinKNOW v20.10.3 with real-time basecalling using Guppy v4.2.2., Sequencing adapters were trimmed using Porechop v0.2.4 from quality filtered reads and the reads were m…","""GPL24106""","""Umashankar,,Singh""","""HoMeCell 

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE197247 m6A MeRIP-seq

- validate date: 2026-02-22
- [m6A modification confers thermal vulnerability to HPV E7 oncotranscripts via reverse regulation of its reader protein IGF2BP1 upon heat stress](https://doi.org/10.1016/j.celrep.2022.111546)
- 熱ストレスとコントロールのデータが揃っている
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動

In [71]:
GSE197247_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE197247/GSE197247_PEP/GSE197247_PEP_raw.csv",
    separator=','
)

display(GSE197247_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_type,cell_line,treatment,rip_antibody,genome_build,supplementary_files_format_and_content,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str
"""caski_ip_ctrl""",null,null,null,null,null,null,"""CaSki IP ctrl""","""GSM5911571""","""Public on Feb 25 2022""","""Feb 22 2022""","""Feb 25 2022""","""SRA""",1,"""cell line""","""Homo sapiens""",9606,"""total RNA""","""m6A MeRIP-seq""","""GPL20301""","""Lingfang,,Wang""","""3110103748@zju.edu.cn""","""Department of Obstetrics and Gynecology""","""Zhejiang University""","""866 Yuhangtang Road""","""Hangzhou""","""Zhejiang Province""",310006,"""China""","""Illumina HiSeq 4000""","""other""","""transcriptomic""","""RIP-Seq""","""GSM5911571""","""NONE""","""GSE197247""",0,"""SRX14256968""","""epithelial cell""","""CaSki""","""control""","""GenSeq m6A-MeRIP Kit (GenSeq, Beijing, China)""","""hg19""","""excel format: Number of raw junction read""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN26179596""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX14256968"""
"""caski_input_ctrl""",null,null,null,null,null,null,"""CaSki Input ctrl""","""GSM5911572""","""Public on Feb 25 2022""","""Feb 22 2022""","""Feb 25 2022""","""SRA""",1,"""cell line""","""Homo sapiens""",9606,"""total RNA""","""m6A MeRIP-seq""","""GPL20301""","""Lingfang,,Wang""","""3110103748@zju.edu.cn""","""Department of Obstetrics and Gynecology""","""Zhejiang University""","""866 Yuhangtang Road""","""Hangzhou""","""Zhejiang Province""",310006,"""China""","""Illumina HiSeq 4000""","""other""","""transcriptomic""","""RIP-Seq""","""GSM5911572""","""NONE""","""GSE197247""",0,"""SRX14256969""","""epithelial cell""","""CaSki""","""control""","""none (input)""","""hg19""","""excel format: Number of raw junction read""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN26179595""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX14256969"""
"""caski_ip_treat""",null,null,null,null,null,null,"""CaSki IP Treat""","""GSM5911573""","""Public on Feb 25 2022""","""Feb 22 2022""","""Feb 25 2022""","""SRA""",1,"""cell line""","""Homo sapiens""",9606,"""total RNA""","""m6A MeRIP-seq""","""GPL20301""","""Lingfang,,Wang""","""3110103748@zju.edu.cn""","""Department of Obstetrics and Gynecology""","""Zhejiang University""","""866 Yuhangtang Road""","""Hangzhou""","""Zhejiang Province""",310006,"""China""","""Illumina HiSeq 4000""","""other""","""transcriptomic""","""RIP-Seq""","""GSM5911573""","""NONE""","""GSE197247""",0,"""SRX14256970""","""epithelial cell""","""CaSki""","""42C 60min treatment""","""GenSeq m6A-MeRIP Kit (GenSeq, Beijing, China)""","""hg19""","""excel format: Number of raw junction read""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN26179594""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX14256970"""
"""caski_input_treat""",null,null,null,null,null,null,"""CaSki Input Treat""","""GSM5911574""","""Public on Feb 25 2022""","""Feb 22 2022""","""Feb 25 2022""","""SRA""",1,"""cell line""","""Homo sapiens""",9606,"""total RNA""","""m6A MeRIP-seq""","""GPL20301""","""Lingfang,,Wang""","""3110103748@zju.edu.cn""","""Department of Obstetrics and Gynecology""","""Zhejiang University""","""866 Yuhangtang Road""","""Hangzhou

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE200269 PRO-IP-seq

- validate date: 2026-02-22
- [PRO-IP-seq tracks molecular modifications of engaged Pol II complexes at nucleotide resolution](https://doi.org/10.1038/s41467-023-42715-3)
- この`PRO-IP-seq`について、特性をまだ把握できていないので一旦こちらに記載
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)に処理を再度移動 (2026-07-23)

In [72]:
# GSE200269_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE200269/GSE200269_PEP/GSE200269_PEP_raw.csv",
#     separator=','
# )

# pl.Config.set_tbl_rows(32)
# display(GSE200269_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE233582 FLAM-seq

- validate date: 2026-02-23
- [RNA tailing machinery drives amyloidogenic phase transition](https://doi.org/10.1073/pnas.2316734121)
- Full-Length-and-mRNA Sequencing (FLAM-seq) という手法のデータ
- 熱ストレス条件に関しても以下のような条件があるため収集
- `Cells were subjected to heat shock by translocation to a 5% CO2 humidified incubator at 43 °C for indicated times.`
- `FLAM seq was performed as described (38). Nucleolar isolation was performed as described above. RNA was extracted using TRIzol reagent from nucleoli/Abodies and was submitted for pooled RNA to yield a no treatment sample and 60-min heat shock sample.`
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動

In [73]:
GSE233582_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE233582/GSE233582_PEP/GSE233582_PEP_raw.csv",
    separator=','
)

display(GSE233582_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,tissue,cell_line,cell_type,treatment,assembly,library_strategy,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str
"""nucleoli_-rep1""",null,null,null,null,null,null,"""Nucleoli -Rep1""","""GSM7430850""","""Public on Jun 01 2024""","""May 26 2023""","""Jun 01 2024""","""SRA""",1,"""Breast""","""Homo sapiens""",9606,"""polyA RNA""","""All three replicates for each unstresssed and stressed were pooled into one processed file from each peak of interest respectively. Then another processed file was generated containing isolated corr…","""GPL28352""","""Stephen,,Lee""","""stephenlee@med.miami.edu""","""Gautier Bldg, Rm 228""","""Biochemistry & Molecular Biology""","""University of Miami""","""1011 NW 15 st.""","""Miami""","""FL""",33136,"""USA""","""Sequel II""","""other""","""transcriptomic""","""OTHER""","""GSM7430850""","""NONE""","""GSE233582, GSE233585""",0,"""SRX20525451""","""Breast""","""MCF-7""","""Adenocarcinoma""","""Unstressed (37deg)""","""GRCh38.primary_assembly.genome.fa""","""FLAM-seq""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN35439845""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX20525451"""
"""nucleoli_-rep2""",null,null,null,null,null,null,"""Nucleoli -Rep2""","""GSM7430851""","""Public on Jun 01 2024""","""May 26 2023""","""Jun 01 2024""","""SRA""",1,"""Breast""","""Homo sapiens""",9606,"""polyA RNA""","""All three replicates for each unstresssed and stressed were pooled into one processed file from each peak of interest respectively. Then another processed file was generated containing isolated corr…","""GPL28352""","""Stephen,,Lee""","""stephenlee@med.miami.edu""","""Gautier Bldg, Rm 228""","""Biochemistry & Molecular Biology""","""University of Miami""","""1011 NW 15 st.""","""Miami""","""FL""",33136,"""USA""","""Sequel II""","""other""","""transcriptomic""","""OTHER""","""GSM7430851""","""NONE""","""GSE233582, GSE233585""",0,"""SRX20525452""","""Breast""","""MCF-7""","""Adenocarcinoma""","""Unstressed (37deg)""","""GRCh38.primary_assembly.genome.fa""","""FLAM-seq""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN35439844""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX20525452"""
"""nucleoli_-rep3""",null,null,null,null,null,null,"""Nucleoli -Rep3""","""GSM7430852""","""Public on Jun 01 2024""","""May 26 2023""","""Jun 01 2024""","""SRA""",1,"""Breast""","""Homo sapiens""",9606,"""polyA RNA""","""All three replicates for each unstresssed and stressed were pooled into one processed file from each peak of interest respectively. Then another processed file was generated containing isolated corr…","""GPL28352""","""Stephen,,Lee""","""stephenlee@med.miami.edu""","""Gautier Bldg, Rm 228""","""Biochemistry & Molecular Biology""","""University of Miami""","""1011 NW 15 st.""","""Miami""","""FL""",33136,"""USA""","""Sequel II""","""other""","""transcriptomic""","""OTHER""","""GSM7430852""","""NONE""","""GSE233582, GSE233585""",0,"""SRX20525453""","""Breast""","""MCF-7""","""Adenocarcinoma""","""Unstressed (37deg)""","""GRCh38.primary_assembly.genome.fa""","""FLAM-seq""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN35439843""","""https://www.ncbi.nlm.nih.gov/s

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE261857 ISO-seq

- validate date: 2026-02-23
- [Nucleolar Pol II interactome reveals TBPL1, PAF1, and Pol I at intergenic rDNA drive rRNA biogenesis](https://doi.org/10.1038/s41467-024-54002-w)
- 論文に以下のような記載があったため、収集する
- `Cells under standard conditions (37 °C) or sincRNA-hyper-inducing conditions (43 °C for 1 h) were washed with PBS, harvested in solution 1 (0.5 M sucrose, 3 mM MgCl2) at 4 °C, and sonicated on ice for 5 × 10 sec using a microtip probe.`
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動

In [74]:
GSE261857_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE261857/GSE261857_PEP/GSE261857_PEP_raw.csv",
    separator=','
)

display(GSE261857_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,tissue,cell_line,cell_type,treatment,assembly,library_strategy,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str
"""unstressed_nucleoli_rep1""",null,null,null,null,null,null,"""Unstressed nucleoli, Rep1""","""GSM8153108""","""Public on Oct 09 2024""","""Mar 18 2024""","""Oct 09 2024""","""SRA""",1,"""Breast""","""Homo sapiens""",9606,"""total RNA""","""Combined UN alignment counts, UN_allreadsaligned.bam.tdf""","""GPL28352""","""Stephen,,Lee""","""stephenlee@med.miami.edu""","""Gautier Bldg, Rm 228""","""Biochemistry & Molecular Biology""","""University of Miami""","""1011 NW 15 st.""","""Miami""","""FL""",33136,"""USA""","""Sequel II""","""other""","""transcriptomic""","""OTHER""","""GSM8153108""","""NONE""","""GSE261857""",0,"""SRX23985220""","""Breast""","""MCF-7""","""Epithelial""","""Unstressed, maintained at 37C""","""GRCh38.primary_assembly.genome.fa""","""ISO-seq""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN40533214""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX23985220"""
"""unstressed_nucleoli_rep2""",null,null,null,null,null,null,"""Unstressed nucleoli, Rep2""","""GSM8153109""","""Public on Oct 09 2024""","""Mar 18 2024""","""Oct 09 2024""","""SRA""",1,"""Breast""","""Homo sapiens""",9606,"""total RNA""","""Combined UN alignment counts, UN_allreadsaligned.bam.tdf""","""GPL28352""","""Stephen,,Lee""","""stephenlee@med.miami.edu""","""Gautier Bldg, Rm 228""","""Biochemistry & Molecular Biology""","""University of Miami""","""1011 NW 15 st.""","""Miami""","""FL""",33136,"""USA""","""Sequel II""","""other""","""transcriptomic""","""OTHER""","""GSM8153109""","""NONE""","""GSE261857""",0,"""SRX23985221""","""Breast""","""MCF-7""","""Epithelial""","""Unstressed, maintained at 37C""","""GRCh38.primary_assembly.genome.fa""","""ISO-seq""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN40533213""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX23985221"""
"""unstressed_nucleoli_rep3""",null,null,null,null,null,null,"""Unstressed nucleoli, Rep3""","""GSM8153110""","""Public on Oct 09 2024""","""Mar 18 2024""","""Oct 09 2024""","""SRA""",1,"""Breast""","""Homo sapiens""",9606,"""total RNA""","""Combined UN alignment counts, UN_allreadsaligned.bam.tdf""","""GPL28352""","""Stephen,,Lee""","""stephenlee@med.miami.edu""","""Gautier Bldg, Rm 228""","""Biochemistry & Molecular Biology""","""University of Miami""","""1011 NW 15 st.""","""Miami""","""FL""",33136,"""USA""","""Sequel II""","""other""","""transcriptomic""","""OTHER""","""GSM8153110""","""NONE""","""GSE261857""",0,"""SRX23985222""","""Breast""","""MCF-7""","""Epithelial""","""Unstressed, maintained at 37C""","""GRCh38.primary_assembly.genome.fa""","""ISO-seq""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN40533212""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX23985222"""
"""heat_shocked_nucleoli_rep1""",null,null,null,null,null,null,"""Heat shocked nucleoli, Rep1""","""GSM8153111""","""Public on Oct 09 2024""","""Mar 18 2024""","""Oct 09 2024""","""SRA""",1,"""Breast""","""Homo sapiens""",9606,"""total RNA""","""Combined HS alignment counts, HS_allreadsaligned.bam.tdf""","""GPL28352""","""Stephen,,Lee""","""stephenle

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE277764 Nanopore

- validate date: 2026-02-23
- [De novo assembly of nuclear stress bodies rearranges and enhances NFIL3 to restrain acute inflammatory responses](https://doi.org/10.1016/j.cell.2025.05.003)
- sodium arsenite (SA) のデータはフィルタリングして収集する
- 論文に以下のような記述があるため収集する
- `For heat shock (HS) treatment, Human HeLa, THP-1 cells and HUVECs were replaced with pre-heated (42°C) conditioned medium and incubated in a culture incubator with an environmental temperature at 42°C and humidified 5% CO2 for 2 h, then recovered at 37°C for 1 h.`
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動

In [75]:
GSE277764_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE277764/GSE277764_PEP/GSE277764_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_taxid_ch1") == 9606) &
    (~pl.col("sample_title").str.contains("_SA_"))
)

display(GSE277764_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,treatment,assembly,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str
"""nanopore_polya_plus_un_rep1""",null,null,null,null,null,null,"""nanopore_polya_plus_UN_rep1""","""GSM8529146""","""Public on May 26 2025""","""Sep 22 2024""","""May 26 2025""","""SRA""",1,"""HeLa""","""Homo sapiens""",9606,"""polyA RNA""","""GPL26167""","""Li,,Yang""","""liyang_fudan@fudan.edu.cn""","""Institutes of Biological Sciences""","""Fudan University""","""131 Dong-An Road""","""Shanghai""",200032,"""China""","""PromethION""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8529146""","""NONE""","""GSE277764""",0,"""SRX26164613""","""HeLa""","""untreated""","""CHM13 T2T v2.0""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN43878907""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX26164613"""
"""nanopore_polya_plus_un_rep2""",null,null,null,null,null,null,"""nanopore_polya_plus_UN_rep2""","""GSM8529147""","""Public on May 26 2025""","""Sep 22 2024""","""May 26 2025""","""SRA""",1,"""HeLa""","""Homo sapiens""",9606,"""polyA RNA""","""GPL26167""","""Li,,Yang""","""liyang_fudan@fudan.edu.cn""","""Institutes of Biological Sciences""","""Fudan University""","""131 Dong-An Road""","""Shanghai""",200032,"""China""","""PromethION""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8529147""","""NONE""","""GSE277764""",0,"""SRX26164614""","""HeLa""","""untreated""","""CHM13 T2T v2.0""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN43878906""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX26164614"""
"""nanopore_polya_plus_hs_rep1""",null,null,null,null,null,null,"""nanopore_polya_plus_HS_rep1""","""GSM8529148""","""Public on May 26 2025""","""Sep 22 2024""","""May 26 2025""","""SRA""",1,"""HeLa""","""Homo sapiens""",9606,"""polyA RNA""","""GPL26167""","""Li,,Yang""","""liyang_fudan@fudan.edu.cn""","""Institutes of Biological Sciences""","""Fudan University""","""131 Dong-An Road""","""Shanghai""",200032,"""China""","""PromethION""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8529148""","""NONE""","""GSE277764""",0,"""SRX26164615""","""HeLa""","""HS stressed""","""CHM13 T2T v2.0""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN43878905""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX26164615"""
"""nanopore_polya_plus_hs_rep2""",null,null,null,null,null,null,"""nanopore_polya_plus_HS_rep2""","""GSM8529149""","""Public on May 26 2025""","""Sep 22 2024""","""May 26 2025""","""SRA""",1,"""HeLa""","""Homo sapiens""",9606,"""polyA RNA""","""GPL26167""","""Li,,Yang""","""liyang_fudan@fudan.edu.cn""","""Institutes of Biological Sciences""","""Fudan University""","""131 Dong-An Road""","""Shanghai""",200032,"""China""","""PromethION""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8529149""","""NONE""","""GSE277764""",0,"""SRX26164616""","""HeLa""","""HS stressed""","""CHM13 T2T v2.0""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN43878904""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX26164616"""
"""nanopore_ribo_minus_un_rep1""",null,null,null,null,null,null,"""nanopore_ribo_minus_UN_rep1""","""GSM8529152""","""Public on May 26 2025""","""Sep 22 2024""","""May 26 2025""","""SRA""",1,"""HeLa""","""Homo sapiens""",9606,"""total RNA""","""GPL26167""","""Li,,Yang""","""liyang_fu

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE280307 ATAC-seq

- validate date: 2026-02-23
- [De novo assembly of nuclear stress bodies rearranges and enhances NFIL3 to restrain acute inflammatory responses](https://doi.org/10.1016/j.cell.2025.05.003)
- [GSE277764](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE277764)のNanoporeのデータに加えて、上記の論文で登録されているデータ
- [./12_8_metadata_atacseq.ipynb](./12_8_metadata_atacseq.ipynb) に移動

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## GSE280712 (TSA-seq)

- validate date: 2026-02-23
- [De novo assembly of nuclear stress bodies rearranges and enhances NFIL3 to restrain acute inflammatory responses](https://doi.org/10.1016/j.cell.2025.05.003)
- [GSE277764](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE277764)のNanoporeのデータに加えて、上記の論文で登録されているデータ
- 念の為、熱ストレスに関連しているデータがあることを記載しておくと思ったが、コントロール条件下のデータが存在しないため、今回は収集しない

In [76]:
# GSE280712_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE280712/GSE280712_PEP/GSE280712_PEP_raw.csv",
#     separator=','
# ).filter(
#     (pl.col("sample_taxid_ch1") == 9606) &
#     (~pl.col("sample_title").str.contains("_SA_"))
# )

# display(GSE280712_PEP)